### **List of Libraries/Dependencies**

In [1]:
# Document Preprocessor
from lxml import etree
from bs4 import BeautifulSoup
from concurrent.futures import ProcessPoolExecutor, as_completed

# Document Converter
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
import re
import unicodedata
import json
import logging
import time
from datetime import datetime
from collections.abc import Iterable
from collections import defaultdict, Counter
from itertools import product
from pathlib import Path

from docling_core.types.doc.base import ImageRefMode
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.document import ConversionResult
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption, HTMLFormatOption

# Schema Loader
# Cite the libraryyyyyyyyyyyyyyyyyyyyyyyyyyyyyy
from rdflib import RDF, RDFS, OWL, URIRef, Namespace, Literal, Dataset, Graph
from rdflib.namespace import XSD, split_uri
import rdflib
import os

# Hybrid Chunker
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY (DOCLING)
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY (HUGGINGFACE)
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from transformers import AutoTokenizer

# Chunk Processor
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
import spacy

# Triple Extractor
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
from dotenv import load_dotenv, find_dotenv
import openai
import asyncio
import os

# Triple Extractor - Batch Processing
import json
import random

# E-R Normalizer
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Graph Serializer
import copy

# Chunk Vector Store
import pickle

# Community (Cluster) Constructor
import igraph as ig
import leidenalg as la


### **Schema Loader**

##### A. Helper Functions & Variables

In [ ]:
# ---------------------------------------------------------------------
# Jurisynth Custom Namespaces
# ---------------------------------------------------------------------
JS_DATA = Namespace("http://jurisynth/data/")
JS_SOURCE = Namespace("http://jurisynth/source/")


# ---------------------------------------------------------------------
# Jurisynth Custom RDF Classes
# ---------------------------------------------------------------------

JS_ASSERTION = JS_DATA.Assertion
JS_MODIFIER = JS_DATA.Modifier
JS_TABLE = JS_DATA.Table
JS_IMAGE = JS_DATA.Image # Not used in the current version; Placeholder for future work


# ---------------------------------------------------------------------
# Helper Functions
# ---------------------------------------------------------------------

def extract_resources(graph, resource_type):
    """
    Extract resources of a given RDF type.

    Returns:
        resources: URI -> label
        metadata: URI -> metadata dict
    """

    resources = dict()
    metadata = dict()

    resource_types = {
        "class": OWL.Class,
        "object property": OWL.ObjectProperty,
        "datatype property": OWL.DatatypeProperty,
        "datatype": RDFS.Datatype
    }

    chosen_type = resource_types[resource_type]

    deprecated_count = 0

    for uri in set(graph.subjects(RDF.type, chosen_type)):

        if isinstance(uri, rdflib.term.BNode):
            continue

        label = graph.value(uri, RDFS.label)
        comment = graph.value(uri, RDFS.comment)

        label = (
            str(label)
            if label
            else uri.split("#")[-1]
        )

        comment = (
            str(comment)
            if comment
            else ""
        )

        # --------------------------------------------------
        # Detect deprecated resources
        # --------------------------------------------------

        is_deprecated = False

        deprecated = graph.value(uri, OWL.deprecated)

        if deprecated is not None:
            if str(deprecated).casefold() == "true":
                is_deprecated = True

        if "deprecated" in label.casefold():
            is_deprecated = True

        if "deprecated" in comment.casefold():
            is_deprecated = True

        if is_deprecated:
            deprecated_count += 1
            continue

        # --------------------------------------------------
        # Build NLP-friendly resource description
        # --------------------------------------------------

        text = label

        if comment:
            text += ". " + comment

        resources[str(uri)] = label

        metadata[str(uri)] = {
            "type": resource_type,
            "label": label,
            "comment": comment,
            "text": text,
            "domain": (
                set(graph.objects(uri, RDFS.domain))
                if resource_type in {"object property", "datatype property"}
                else set()
            ),
            "range": (
                set(graph.objects(uri, RDFS.range))
                if resource_type in {"object property", "datatype property"}
                else set()
            )
        }

    print(
        f"Filtered {deprecated_count} deprecated "
        f"{resource_type.replace('_', ' ')} resources."
    )

    return resources, metadata

##### B. Execution

In [ ]:
# Point to the schema folder
schema_folder = "./schema"
schema_files = [os.path.join(schema_folder, file) for file in os.listdir(schema_folder)]

schema_graph = Graph()

index = 1
for file in schema_files:
    schema_graph.parse(file, format="xml")
    print(f"Finished parsing RDF file {index}.")
    index += 1

print()

# Only retrieves declared namespaces
schema_namespaces = set(schema_graph.namespaces())
# schema_namespaces.remove(('', rdflib.term.URIRef('http://publications.europa.eu/ontology/cdmplus#')))
# schema_namespaces.add(('cdmplus#', rdflib.term.URIRef('http://publications.europa.eu/ontology/cdmplus#')))

print("Schema Namespaces:")
for ns in schema_namespaces:
    print(ns)

print()

# Classes
classes, class_metadata = extract_resources(schema_graph, "class")
print(f"Found {len(classes)} classes.")

# Object properties
obj_properties, obj_metadata = extract_resources(schema_graph, "object property")
print(f"Found {len(obj_properties)} object properties/relations.")

# Datatype properties
datatype_properties, data_prop_metadata = extract_resources(schema_graph, "datatype property")
print(f"Found {len(datatype_properties)} datatype properties/relations.")

# Datatypes
datatypes, datatype_metadata = extract_resources(schema_graph, "datatype")
print(f"Found {len(datatypes)} datatypes.")

# Resource Metadata
resource_metadata = dict()

resource_metadata.update(class_metadata)
resource_metadata.update(obj_metadata)
resource_metadata.update(data_prop_metadata)
resource_metadata.update(datatype_metadata)

rdf_resources = list(classes.keys()) + list(obj_properties.keys()) + list(datatype_properties.keys()) + list(datatypes.keys())
rdf_dict = {**classes, **obj_properties, **datatype_properties, **datatypes}


Finished parsing RDF file 1.
Finished parsing RDF file 2.

Schema Namespaces:
('rdf', rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#'))
('vann', rdflib.term.URIRef('http://purl.org/vocab/vann/'))
('dcat', rdflib.term.URIRef('http://www.w3.org/ns/dcat#'))
('dc', rdflib.term.URIRef('http://purl.org/dc/elements/1.1/'))
('prov', rdflib.term.URIRef('http://www.w3.org/ns/prov#'))
('cidoc-crm', rdflib.term.URIRef('http://www.cidoc-crm.org/cidoc-crm/'))
('owl', rdflib.term.URIRef('http://www.w3.org/2002/07/owl#'))
('skos', rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#'))
('sosa', rdflib.term.URIRef('http://www.w3.org/ns/sosa/'))
('admin', rdflib.term.URIRef('http://publications.europa.eu/ontology/cdm/admin#'))
('doap', rdflib.term.URIRef('http://usefulinc.com/ns/doap#'))
('org', rdflib.term.URIRef('http://www.w3.org/ns/org#'))
('cmr', rdflib.term.URIRef('http://publications.europa.eu/ontology/cdm/cmr#'))
('xml', rdflib.term.URIRef('http://www.w3.org/XML/1998/namespa

##### C. Inspection

In [ ]:
resource_metadata

{'http://publications.europa.eu/ontology/cdm#asset-classification': {'type': 'class',
  'label': 'Concept representing an asset classification',
  'comment': 'Classification of an asset (concepts from AT asset-classification)',
  'text': 'Concept representing an asset classification. Classification of an asset (concepts from AT asset-classification)',
  'domain': set(),
  'range': set()},
 'http://publications.europa.eu/ontology/cdm#event_legal': {'type': 'class',
  'label': 'Legal event',
  'comment': 'Any activity in a legislative process involving a recognizable author and often a recipient . The event types are based on concepts in AT event.',
  'text': 'Legal event. Any activity in a legislative process involving a recognizable author and often a recipient . The event types are based on concepts in AT event.',
  'domain': set(),
  'range': set()},
 'http://publications.europa.eu/ontology/cdm#opinion_ep': {'type': 'class',
  'label': 'European Parliament opinion',
  'comment': '',


In [ ]:
print("No. of RDF Resources:", len(rdf_resources))

No. of RDF Resources: 2625


### **Document Preprocessor**

##### A. Execution

In [ ]:
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from time import perf_counter
from sentence_transformers import SentenceTransformer
import importlib
import worker


# =============================================================================
# Configuration
# =============================================================================

MAX_WORKERS = 6
BATCH_SIZE = 128

DOCS_DIR = Path("big_samples")

embed_model = SentenceTransformer("all-MiniLM-L6-v2")


# =============================================================================
# Locate benchmark documents
# =============================================================================

benchmark_files = sorted(
    DOCS_DIR.glob("*.html")
)

if not benchmark_files:
    raise FileNotFoundError(
        f"No HTML files found in: {DOCS_DIR.resolve()}"
    )

# ---------------------------------------------------------------------
# Canonical document IDs
# ---------------------------------------------------------------------

doc_ids = {
    doc_path: doc_path.stem
    for doc_path in benchmark_files
}

print(f"Initialized {len(doc_ids):,} document IDs.")


# =============================================================================
# Reload latest worker.py
# =============================================================================

worker = importlib.reload(worker)


# =============================================================================
# Header
# =============================================================================

print("=" * 80)
print("FULL TABLE PIPELINE BENCHMARK")
print("=" * 80)
print(f"Documents       : {len(benchmark_files):,}")
print(f"max_workers     : {MAX_WORKERS}")
print(f"embedding batch : {BATCH_SIZE}")
print(f"Input directory : {DOCS_DIR.resolve()}")
print("=" * 80)


# =============================================================================
# Temporary benchmark outputs
# =============================================================================
#
# Prevent this benchmark from overwriting your real table_store or
# processed_docs directories.
# =============================================================================

table_store = Path("table_store")
processed_dir = Path("processed_docs")

table_store.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------------
# Phase 1: document/table extraction
# -------------------------------------------------------------------------

print()
print("=" * 80)
print("PHASE 1 — HTML / TABLE EXTRACTION")
print("=" * 80)

phase1_start = perf_counter()

completed = 0
results = []

with ProcessPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = {
        executor.submit(
            worker.extract_only,
            path,
            doc_ids[path],
            str(table_store),
            str(processed_dir),
        ): path
        for path in benchmark_files
    }

    for future in as_completed(futures):

        source_path = futures[future]

        try:
            result = future.result()

        except Exception as exc:

            print(
                f"[ERROR] {source_path.name}: "
                f"{type(exc).__name__}: {exc}",
                flush=True,
            )

            raise

        results.append(result)
        completed += 1

        diagnostics = result["diagnostics"]

        phase1_elapsed = (
            perf_counter() - phase1_start
        )

        rate = (
            completed / phase1_elapsed
            if phase1_elapsed > 0
            else 0
        )

        remaining = (
            len(benchmark_files)
            - completed
        )

        eta = (
            remaining / rate
            if rate > 0
            else 0
        )

        print(
            f"[{completed:02d}/{len(benchmark_files):02d}] "
            f"{source_path.name:<45} "
            f"| semantic={diagnostics['semantic_tables']:4d} "
            f"| rows={diagnostics['semantic_rows']:7,d} "
            f"| headerless={diagnostics['headerless_semantic_tables']:3d} "
            f"| {phase1_elapsed:7.1f}s "
            f"| ETA={eta:6.1f}s",
            flush=True,
        )

phase1_elapsed = (
    perf_counter() - phase1_start
)

print()
print(
    f"Phase 1 complete: "
    f"{phase1_elapsed:.2f}s"
)


# -------------------------------------------------------------------------
# Collect semantic tables
# -------------------------------------------------------------------------

all_tables = []

for result in results:
    all_tables.extend(
        result.get("tables") or []
    )

print(
    f"Semantic tables collected: "
    f"{len(all_tables):,}"
)


# -------------------------------------------------------------------------
# Phase 2: build FAISS table/row index
# -------------------------------------------------------------------------

print()
print("=" * 80)
print("PHASE 2 — TABLE / ROW EMBEDDING + FAISS INDEXING")
print("=" * 80)

phase2_start = perf_counter()

retrieval_index = worker.build_table_index(
    all_tables,
    embed_model,
    batch_size=BATCH_SIZE,
)

phase2_elapsed = (
    perf_counter() - phase2_start
)

print()
print(
    f"Phase 2 complete: "
    f"{phase2_elapsed:.2f}s"
)


# -------------------------------------------------------------------------
# Total pipeline
# -------------------------------------------------------------------------

total_elapsed = (
    phase1_elapsed
    + phase2_elapsed
)


# =============================================================================
# Diagnostics
# =============================================================================

total_source = sum(
    r["diagnostics"]["source_tables"]
    for r in results
)

total_sparse = sum(
    r["diagnostics"]["sparse_tables"]
    for r in results
)

total_candidate = sum(
    r["diagnostics"]["candidate_tables"]
    for r in results
)

total_semantic = sum(
    r["diagnostics"]["semantic_tables"]
    for r in results
)

total_rows = sum(
    r["diagnostics"]["semantic_rows"]
    for r in results
)

total_headerless = sum(
    r["diagnostics"]["headerless_semantic_tables"]
    for r in results
)

total_merged_fragments = sum(
    r["diagnostics"]["merged_fragments"]
    for r in results
)


# =============================================================================
# Throughput / 40K projection
# =============================================================================

docs = len(results)

docs_per_second = (
    docs / total_elapsed
    if total_elapsed > 0
    else 0
)

seconds_per_doc = (
    total_elapsed / docs
    if docs > 0
    else 0
)

estimated_40k_seconds = (
    40_000 / docs_per_second
    if docs_per_second > 0
    else float("inf")
)

estimated_40k_hours = (
    estimated_40k_seconds / 3600
)

estimated_40k_days = (
    estimated_40k_hours / 24
)


# =============================================================================
# Final report
# =============================================================================

print()
print("=" * 80)
print("FINAL RESULTS")
print("=" * 80)

print()
print("TIMING")
print("-" * 80)
print(
    f"Phase 1 — extraction       : "
    f"{phase1_elapsed:,.2f}s"
)
print(
    f"Phase 2 — indexing         : "
    f"{phase2_elapsed:,.2f}s"
)
print(
    f"TOTAL PIPELINE             : "
    f"{total_elapsed:,.2f}s"
)
print(
    f"Average / document         : "
    f"{seconds_per_doc:.2f}s"
)
print(
    f"Overall throughput         : "
    f"{docs_per_second:.3f} docs/s"
)

print()
print("TABLE DIAGNOSTICS")
print("-" * 80)
print(
    f"Source tables              : "
    f"{total_source:,}"
)
print(
    f"Sparse tables              : "
    f"{total_sparse:,}"
)
print(
    f"Candidate tables           : "
    f"{total_candidate:,}"
)
print(
    f"Semantic tables            : "
    f"{total_semantic:,}"
)
print(
    f"Semantic rows              : "
    f"{total_rows:,}"
)
print(
    f"Headerless semantic tables: "
    f"{total_headerless:,}"
)
print(
    f"Merged fragments           : "
    f"{total_merged_fragments:,}"
)

print()
print("INDEX")
print("-" * 80)

if retrieval_index["table_index"] is not None:
    print(
        f"Table vectors              : "
        f"{retrieval_index['table_index'].ntotal:,}"
    )
else:
    print("Table vectors              : 0")

total_row_vectors = sum(
    index.ntotal
    for index in retrieval_index[
        "row_indices"
    ].values()
)

print(
    f"Row vectors                : "
    f"{total_row_vectors:,}"
)

print(
    f"Row indices                : "
    f"{len(retrieval_index['row_indices']):,}"
)

print()
print("=" * 80)
print("40,000-DOCUMENT PROJECTION")
print("=" * 80)

print(
    f"Estimated total time       : "
    f"{estimated_40k_seconds:,.0f}s"
)

print(
    f"Estimated total time       : "
    f"{estimated_40k_hours:,.2f}h"
)

print(
    f"Estimated total time       : "
    f"{estimated_40k_days:,.2f} days"
)

print("=" * 80)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FULL TABLE PIPELINE BENCHMARK
Documents       : 30
max_workers     : 6
embedding batch : 128
Input directory : C:\Users\Roxas\OneDrive\Desktop\Project_Space\prototype\big_samples

PHASE 1 — HTML / TABLE EXTRACTION
[01/30] L_2005073EN.01000101.html                     | semantic=   0 | rows=      0 | headerless=  0 |     1.0s | ETA=  30.4s
[02/30] L_2005265EN.01000101.html                     | semantic=  15 | rows=  1,920 | headerless=  0 |     4.2s | ETA=  59.4s
[03/30] C_2022077EN.01001901.html                     | semantic=  10 | rows=  3,684 | headerless=  0 |     5.2s | ETA=  46.7s
[04/30] L_1994001EN.01000101.html                     | semantic=  20 | rows=    836 | headerless=  2 |     5.4s | ETA=  34.9s
[05/30] L_2006302EN.01001001.html                     | semantic=   0 | rows=      0 | headerless=  0 |     5.4s | ETA=  27.1s
[06/30] L_2004304EN.01003801.html                     | semantic=  13 | rows=  1,152 | headerless=  1 |     5.6s | ETA=  22.5s
[07/30] L_2006399EN.0100

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Row-level units: 247,384


Batches:   0%|          | 0/1933 [00:00<?, ?it/s]

Tables indexed : 1,183
Row indices    : 1,183
Rows indexed   : 247,384

Phase 2 complete: 1292.29s

FINAL RESULTS

TIMING
--------------------------------------------------------------------------------
Phase 1 — extraction       : 33.65s
Phase 2 — indexing         : 1,292.29s
TOTAL PIPELINE             : 1,325.94s
Average / document         : 44.20s
Overall throughput         : 0.023 docs/s

TABLE DIAGNOSTICS
--------------------------------------------------------------------------------
Source tables              : 18,585
Sparse tables              : 17,400
Candidate tables           : 1,185
Semantic tables            : 1,183
Semantic rows              : 247,384
Headerless semantic tables: 59
Merged fragments           : 2

INDEX
--------------------------------------------------------------------------------
Table vectors              : 1,183
Row vectors                : 247,384
Row indices                : 1,183

40,000-DOCUMENT PROJECTION
Estimated total time       : 1,767,924s
E

In [ ]:
from rdflib import Literal, RDF
from worker import generate_anchor_sentence

# =============================================================================
# RDF Table Instantiation
# =============================================================================

def instantiate_table_resources(
    extraction_results,
    graph,
):
    """
    Instantiate extracted semantic tables as RDF resources.

    One RDF resource is created per semantic table.
    The exact table contents remain in the JSON table store and are
    intentionally not expanded into RDF rows/cells.

    Returns:
        dict:
            (doc_id, table_id) -> table URI
    """

    table_resources = dict()

    for result in extraction_results:

        doc_id = result["doc_id"]
        tables = result.get("tables") or list()

        source_uri = JS_SOURCE[doc_id]

        for table in tables:

            table_id = table["table_id"]

            table_uri = JS_DATA[
                f"table_{doc_id}_{table_id}"
            ]

            # -------------------------------------------------------------
            # Table identity
            # -------------------------------------------------------------

            graph.add((
                table_uri,
                RDF.type,
                JS_TABLE,
            ))

            # -------------------------------------------------------------
            # Source document
            # -------------------------------------------------------------

            graph.add((
                table_uri,
                JS_DATA.sourceDocument,
                source_uri,
            ))

            # -------------------------------------------------------------
            # Document context
            # -------------------------------------------------------------

            context = table.get("context")

            if context:
                graph.add((
                    table_uri,
                    JS_DATA.context,
                    Literal(context),
                ))

            # -------------------------------------------------------------
            # Retrieval description
            # -------------------------------------------------------------

            description = (
                table.get("description")
                or generate_anchor_sentence(table)
            )

            if description:
                graph.add((
                    table_uri,
                    JS_DATA.description,
                    Literal(description),
                ))

            table_resources[
                (doc_id, table_id)
            ] = table_uri

    return table_resources


# Instantiate all extracted semantic tables
table_resources = instantiate_table_resources(
    extraction_results,
    graph,
)

print(
    f"Instantiated {len(table_resources):,} RDF Table resources."
)

NameError: name 'extraction_results' is not defined

##### B. Inspection

In [8]:
# =============================================================================
# CURRENT TABLE-PROCESSOR INSPECTION
# =============================================================================

from pathlib import Path
import random
import time

# Import the actual worker logic so this diagnostic cannot drift away from it.
from worker import (
    triage_and_extract,
    merge_fragments,
)


# =============================================================================
# Configuration
# =============================================================================

INPUT_DIR = Path(
    r"C:\Users\Roxas\OneDrive\Desktop\Project_Space\prototype\big_samples"
)

DOC_SAMPLE_SIZE = 10
EXAMPLES_PER_CATEGORY = 3
MAX_DISPLAY_ROWS = 4


# =============================================================================
# Small diagnostic helpers
# =============================================================================

def has_explicit_header_html(table):
    """Check whether the current HTML table explicitly uses <th>."""

    return any(
        tr.find("th", recursive=False)
        for tr in table.find_all("tr")
        if tr.find_parent("table") is table
    )


def has_bold_first_row_html(table):
    """Check whether every first-row cell contains <b> or <strong>."""

    first_tr = table.find("tr")

    if not first_tr:
        return False

    cells = first_tr.find_all(
        ["td", "th"],
        recursive=False,
    )

    return (
        bool(cells)
        and all(
            cell.find(["b", "strong"])
            for cell in cells
        )
    )


def print_rows(rows, max_rows=MAX_DISPLAY_ROWS):
    """Print a compact preview of table rows."""

    for i, row in enumerate(rows[:max_rows]):
        print(
            f"    [{i}] "
            + " | ".join(
                str(cell).strip()
                for cell in row
            )
        )

    if len(rows) > max_rows:
        print(
            f"    ... {len(rows) - max_rows} more rows"
        )


def print_example(
    category,
    html_path,
    table,
    table_id,
    header,
    data,
):
    """Print one concise representative example."""

    print()
    print(
        f"[{category}] "
        f"{html_path.name} | "
        f"table={table_id}"
    )

    print(
        f"  rows={len(data) + (1 if header else 0)} | "
        f"cols={len(header) if header else (len(data[0]) if data else 0)} | "
        f"explicit_th={has_explicit_header_html(table)} | "
        f"bold_first={has_bold_first_row_html(table)} | "
        f"inferred_header={header is not None}"
    )

    if header:
        print(
            "  HEADER: "
            + " | ".join(
                str(cell).strip()
                for cell in header
            )
        )

    print("  ROWS:")
    print_rows(data)


# =============================================================================
# Main inspection
# =============================================================================

html_files = sorted(
    INPUT_DIR.glob("*.html")
)

print("=" * 80)
print("CURRENT TABLE-PROCESSOR INSPECTION")
print("=" * 80)
print(
    f"Documents: {len(html_files)}"
)
print("=" * 80)


# =============================================================================
# Scan
# =============================================================================

document_results = []

totals = {
    "source_tables": 0,
    "candidate_tables": 0,
    "sparse_tables": 0,
    "semantic_tables": 0,
    "semantic_rows": 0,
    "headered_tables": 0,
    "headerless_tables": 0,
    "explicit_th": 0,
    "bold_first": 0,
    "merged_tables": 0,
    "merged_fragments": 0,
    "merged_headerless": 0,
}

start_time = time.time()


for doc_index, html_path in enumerate(
    html_files,
    1,
):

    print(
        f"[{doc_index:02d}/{len(html_files):02d}] "
        f"{html_path.name} ...",
        flush=True,
    )

    try:

        results, soup = triage_and_extract(
            html_path
        )

        merged = merge_fragments(
            results,
            html_path.stem,
        )

    except Exception as exc:

        import traceback

        print(
            f"    ERROR: {type(exc).__name__}: {exc}",
            flush=True,
        )

        traceback.print_exc()

        continue

    source_tables = len(results)

    candidate_tables = sum(
        info["status"] == "candidate"
        for info in results
    )

    sparse_tables = sum(
        info["status"] == "sparse"
        for info in results
    )

    headered = sum(
        info.get("header") is not None
        for info in results
        if info["status"] == "candidate"
    )

    headerless = (
        candidate_tables - headered
    )

    explicit_th = sum(
        has_explicit_header_html(info["tag"])
        for info in results
    )

    bold_first = sum(
        has_bold_first_row_html(info["tag"])
        for info in results
    )

    semantic_rows = sum(
        len(table.get("data") or [])
        for table in merged
    )

    merged_tables = sum(
        len(table.get("tags") or []) > 1
        for table in merged
    )

    merged_fragments = sum(
        max(
            len(table.get("tags") or []) - 1,
            0,
        )
        for table in merged
    )

    merged_headerless = sum(
        (
            len(table.get("tags") or []) > 1
            and table.get("header") is None
        )
        for table in merged
    )

    document_results.append({
        "path": html_path,
        "results": results,
        "merged": merged,
    })

    totals["source_tables"] += source_tables
    totals["candidate_tables"] += candidate_tables
    totals["sparse_tables"] += sparse_tables
    totals["semantic_tables"] += len(merged)
    totals["semantic_rows"] += semantic_rows
    totals["headered_tables"] += headered
    totals["headerless_tables"] += headerless
    totals["explicit_th"] += explicit_th
    totals["bold_first"] += bold_first
    totals["merged_tables"] += merged_tables
    totals["merged_fragments"] += merged_fragments
    totals["merged_headerless"] += merged_headerless

    print(
        f"    source={source_tables:,} | "
        f"candidate={candidate_tables:,} | "
        f"semantic={len(merged):,} | "
        f"headered={headered:,} | "
        f"headerless={headerless:,} | "
        f"merged_fragments={merged_fragments:,}",
        flush=True,
    )


elapsed = time.time() - start_time


# =============================================================================
# Overall summary
# =============================================================================

print()
print("=" * 80)
print("OVERALL")
print("=" * 80)

print(
    f"source tables       : "
    f"{totals['source_tables']:,}"
)

print(
    f"candidate tables    : "
    f"{totals['candidate_tables']:,}"
)

print(
    f"semantic tables     : "
    f"{totals['semantic_tables']:,}"
)

print(
    f"semantic rows       : "
    f"{totals['semantic_rows']:,}"
)

print(
    f"headered            : "
    f"{totals['headered_tables']:,}"
)

print(
    f"headerless          : "
    f"{totals['headerless_tables']:,}"
)

print(
    f"explicit <th>       : "
    f"{totals['explicit_th']:,}"
)

print(
    f"bold first row      : "
    f"{totals['bold_first']:,}"
)

print(
    f"merged tables       : "
    f"{totals['merged_tables']:,}"
)

print(
    f"merged fragments    : "
    f"{totals['merged_fragments']:,}"
)

print(
    f"merged headerless   : "
    f"{totals['merged_headerless']:,}"
)

print(
    f"elapsed             : "
    f"{elapsed:.1f}s"
)

print("=" * 80)


# =============================================================================
# Representative examples
# =============================================================================

categories = {
    "HEADERED": [],
    "HEADERLESS": [],
    "MERGED": [],
    "MERGED_HEADERLESS": [],
}


for doc in document_results:

    html_path = doc["path"]
    results = doc["results"]
    merged = doc["merged"]

    # -------------------------------------------------------------
    # Headered / headerless examples from actual triage output
    # -------------------------------------------------------------

    for info in results:

        if info["status"] != "candidate":
            continue

        header = info.get("header")
        data = info.get("data") or []

        if header is not None:
            categories["HEADERED"].append(
                (
                    html_path,
                    info["tag"],
                    info.get("table_id"),
                    header,
                    data,
                )
            )

        else:
            categories["HEADERLESS"].append(
                (
                    html_path,
                    info["tag"],
                    info.get("table_id"),
                    None,
                    data,
                )
            )

    # -------------------------------------------------------------
    # Merged examples
    # -------------------------------------------------------------

    for table in merged:

        if len(table.get("tags") or []) <= 1:
            continue

        category = (
            "MERGED_HEADERLESS"
            if table.get("header") is None
            else "MERGED"
        )

        categories[category].append(
            (
                html_path,
                table["tags"][0],
                table["table_id"],
                table.get("header"),
                table.get("data") or [],
            )
        )


# =============================================================================
# Print representative examples
# =============================================================================

print()
print("=" * 80)
print("REPRESENTATIVE EXAMPLES")
print("=" * 80)


rng = random.Random(42)


for category, examples in categories.items():

    print()
    print(
        f"[{category}] "
        f"available={len(examples):,}"
    )

    if not examples:
        print("  None.")
        continue

    selected = (
        rng.sample(
            examples,
            min(
                EXAMPLES_PER_CATEGORY,
                len(examples),
            ),
        )
    )

    for (
        html_path,
        table,
        table_id,
        header,
        data,
    ) in selected:

        print_example(
            category,
            html_path,
            table,
            table_id,
            header,
            data,
        )


print()
print("=" * 80)
print("INSPECTION FINISHED")
print("=" * 80)

CURRENT TABLE-PROCESSOR INSPECTION
Documents: 30
[01/30] C_2021302EN.01000101.html ...
    source=58 | candidate=26 | semantic=26 | headered=26 | headerless=0 | merged_fragments=0
[02/30] C_2022077EN.01001901.html ...
    source=26 | candidate=10 | semantic=10 | headered=10 | headerless=0 | merged_fragments=0
[03/30] L_1987256EN.01000101.html ...
    source=770 | candidate=102 | semantic=102 | headered=102 | headerless=0 | merged_fragments=0
[04/30] L_1994001EN.01000101.html ...
    source=1,976 | candidate=20 | semantic=20 | headered=18 | headerless=2 | merged_fragments=0
[05/30] L_2004304EN.01003801.html ...
    source=5,509 | candidate=13 | semantic=13 | headered=12 | headerless=1 | merged_fragments=0
[06/30] L_2005073EN.01000101.html ...
    source=9 | candidate=0 | semantic=0 | headered=0 | headerless=0 | merged_fragments=0
[07/30] L_2005265EN.01000101.html ...
    source=460 | candidate=15 | semantic=15 | headered=15 | headerless=0 | merged_fragments=0
[08/30] L_2006301EN.0100010

### **Document Converter**

##### A. Helper Functions & Variables

In [5]:
# Cleans documents
def clean_text(text: str) -> str:
    if not text:
        return text

    # 1. Normalize Unicode (fixes odd composed characters)
    text = unicodedata.normalize("NFKC", text)

    # 2. Replace the following:
    replacements = {
        "\u00A0": " ",  # NBSP
        "\u202F": " ",  # narrow NBSP
        "\u202f": " ",  # narrow NBSP
        "\u2009": " ",  # thin space
        "\u2007": " ",  # figure space
        "\x00": "",     # null bytes
        "\ufffd": ""    # Unicode replacement char
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    # 4. Remove control characters (but keep newlines/tabs)
    text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", text)

    return text

_log = logging.getLogger(__name__)

# Export toggles:
# - USE_V2 controls modern Docling document exports.
# - USE_LEGACY enables legacy Deep Search exports for comparison or migration.
USE_V2 = True
USE_LEGACY = False


def export_documents(
    conv_results: Iterable[ConversionResult],
    output_dir: Path,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    failure_count = 0
    partial_success_count = 0

    for conv_res in conv_results:
        if conv_res.status == ConversionStatus.SUCCESS:
            success_count += 1
            doc_filename = conv_res.input.file.stem

            if USE_V2:
                # Export converted files as markdown files
                conv_res.document.save_as_markdown(
                    output_dir / f"{doc_filename}.md",
                    image_mode=ImageRefMode.PLACEHOLDER,
                )
                
                # conv_res.document.save_as_json(
                #     output_dir / f"{doc_filename}.json",
                #     image_mode=ImageRefMode.PLACEHOLDER,
                # )
                
                # conv_res.document.save_as_markdown(
                #     output_dir / f"{doc_filename}.txt",
                #     image_mode=ImageRefMode.PLACEHOLDER,
                #     strict_text=True,
                # )

                # Export Docling document format to markdown:
                with (output_dir / f"{doc_filename}.md").open("w", encoding="utf-8") as fp:
                    raw_md = conv_res.document.export_to_markdown()
                    fp.write(clean_text(raw_md))

                # # Export Docling document format to text:
                # with (output_dir / f"{doc_filename}.txt").open("w") as fp:
                #     fp.write(conv_res.document.export_to_markdown(strict_text=True))

            if USE_LEGACY:
                # Export Markdown format:
                with (output_dir / f"{doc_filename}.legacy.md").open("w", encoding="utf-8") as fp:
                    raw_md = conv_res.document.export_to_markdown()
                    fp.write(clean_text(raw_md))

                # # Export Deep Search document JSON format:
                # with (output_dir / f"{doc_filename}.legacy.json").open(
                #     "w", encoding="utf-8"
                # ) as fp:
                #     fp.write(json.dumps(conv_res.document.export_to_dict()))

                # # Export Text format:
                # with (output_dir / f"{doc_filename}.legacy.txt").open(
                #     "w", encoding="utf-8"
                # ) as fp:
                #     fp.write(
                #         conv_res.document.export_to_markdown(strict_text=True)
                #     )

                # # Export Document Tags format:
                # with (output_dir / f"{doc_filename}.legacy.doctags.txt").open(
                #     "w", encoding="utf-8"
                # ) as fp:
                #     fp.write(conv_res.document.export_to_doctags())

        elif conv_res.status == ConversionStatus.PARTIAL_SUCCESS:
            _log.info(
                f"Document {conv_res.input.file} was partially converted with the following errors:"
            )
            for item in conv_res.errors:
                _log.info(f"\t{item.error_message}")
            partial_success_count += 1
        else:
            _log.info(f"Document {conv_res.input.file} failed to convert.")
            failure_count += 1

    _log.info(
        f"Processed {success_count + partial_success_count + failure_count} docs, "
        f"of which {failure_count} failed "
        f"and {partial_success_count} were partially converted."
    )
    return success_count, partial_success_count, failure_count


def convert_documents(folder_path):
    logging.basicConfig(level=logging.INFO)

    # Location of source documents
    data_folder = folder_path
    input_doc_paths = [file_path for file_path in data_folder.iterdir()]

    # buf = BytesIO((data_folder / "pdf/2206.01062.pdf").open("rb").read())
    # docs = [DocumentStream(name="my_doc.pdf", stream=buf)]
    # input = DocumentConversionInput.from_streams(docs)

    # # Turn on inline debug visualizations:
    # settings.debug.visualize_layout = True
    # settings.debug.visualize_ocr = True
    # settings.debug.visualize_tables = True
    # settings.debug.visualize_cells = True

    # Configure the PDF pipeline. Enabling page image generation improves HTML
    # previews (embedded images) but adds processing time.
    pdf_pipeline_options = PdfPipelineOptions()
    pdf_pipeline_options.generate_page_images = True

    doc_converter = DocumentConverter(
        allowed_formats=[
            InputFormat.PDF,
            InputFormat.HTML
        ],
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pdf_pipeline_options,
                backend=DoclingParseV4DocumentBackend
            ),
            InputFormat.HTML: HTMLFormatOption()
        }
    )

    start_time = time.time()

    # Convert all inputs. Set `raises_on_error=False` to keep processing other
    # files even if one fails; errors are summarized after the run.
    conv_results = doc_converter.convert_all(
        input_doc_paths,
        raises_on_error=False,  # to let conversion run through all and examine results at the end
    )
    # Write outputs to ./scratch and log a summary.
    _success_count, _partial_success_count, failure_count = export_documents(
        conv_results, output_dir=Path("converted_docs")
    )

    end_time = time.time() - start_time

    _log.info(f"Document conversion complete in {end_time:.2f} seconds.")

    if failure_count > 0:
        raise RuntimeError(
            f"The example failed converting {failure_count} on {len(input_doc_paths)}."
        )

##### B. Execution

In [ ]:
batch_folder_path = Path("./processed_docs")

convert_documents(batch_folder_path)

INFO:docling.datamodel.document:detected formats: [<InputFormat.HTML: 'html'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for SimplePipeline with options hash 7d306d2d021deac65a97d1a5f925362a
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered picture descriptions: ['picture_description_vlm_engine', 'vlm', 'api']
INFO:docling.pipeline.base_pipeline:Processing document 31953D0030en.html
INFO:docling.document_converter:Finished converting document 31953D0030en.html in 0.14 sec.
INFO:docling.datamodel.document:detected formats: [<InputFormat.HTML: 'html'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.pipeline.base_pipeline:Processing document 31954S0024en.html
INFO:docling.document_converter:Finished converting document 31954S0024en.html in 0.03 sec.
INFO:docling.datamodel.document:detected formats: [<InputFormat

### **Hybrid Chunker**

##### A. Execution

In [ ]:
# Options:
# "openai/gpt-oss-120b"
# nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16
EMBED_MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16"

MAX_TOKENS = 1024

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID),
    max_tokens=MAX_TOKENS,
)

chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True,
)

digitalizer = DocumentConverter()

conv_docs_folder = Path("./converted_docs")
conv_doc_paths = [file_path for file_path in conv_docs_folder.iterdir()]

raw_chunks = list()

for doc_path in conv_doc_paths:

    doc_id = doc_ids[doc_path]
    doc = digitalizer.convert(source=doc_path).document
    chunks = chunker.chunk(dl_doc=doc)

    for i, chunk in enumerate(chunks):
        raw_chunks.append(
            {
                "doc_id": doc_id,
                "chunk_id": f"chunk_{i + 1}",
                "chunk": chunk
            }
        )

INFO:httpx:HTTP Request: HEAD https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16/624ba927cfbef0427354998700de3d51173c8c04/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16/624ba927cfbef0427354998700de3d51173c8c04/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/nvidia/NVIDI

##### B. Chunk Inspection

In [13]:
# Pick the first document
sample_file = raw_chunks[0]["doc_id"]

# Get all chunks belonging to that document
sample_chunk_list = [
    item
    for item in raw_chunks
    if item["doc_id"] == sample_file
]

print("The current chunks are from:", sample_file, "\n")

for item in sample_chunk_list:
    chunk_id = item["chunk_id"]
    chunk = item["chunk"]

    print(f"=== {chunk_id} ===")

    txt_tokens = tokenizer.count_tokens(chunk.text)
    print(f"chunk.text ({txt_tokens} tokens):\n{chunk.text!r}")

    ser_txt = chunker.contextualize(chunk=chunk)
    ser_tokens = tokenizer.count_tokens(ser_txt)
    print(
        f"chunker.contextualize(chunk) ({ser_tokens} tokens):\n{ser_txt!r}"
    )

    print()

The current chunks are from: 31953D0030en.md 

=== chunk_1 ===
chunk.text (236 tokens):
'**ECSC High Authority: Decision No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel**\n*Official Journal 006 , 04/05/1953 P. 0109 - 0110*\n*Danish special edition: Series I Chapter 1952-1958 P. 0009*\n*English special edition: Series I Chapter 1952-1958 P. 0009*\n*Greek special edition: Chapter 08 Volume 1 P. 0005*\n*Spanish special edition: Chapter 08 Volume 1 P. 0005*\n*Portuguese special edition Chapter 08 Volume 1 P. 0005*\n*Finnish special edition: Chapter 12 Volume 3 P. 0003*\n*Swedish special edition: Chapter 12 Volume 3 P. 0003*'
chunker.contextualize(chunk) (247 tokens):
'31953D0030\n**ECSC High Authority: Decision No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel**\n*Official Journal 006 , 04/05/1953 P. 0109 - 0110*\n*Danish special edition: Series I Chap

In [14]:
len(raw_chunks)

104

In [15]:
raw_chunks[0]

{'doc_id': '31953D0030en.md',
 'chunk_id': 'chunk_1',
 'chunk': DocChunk(text='**ECSC High Authority: Decision No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel**\n*Official Journal 006 , 04/05/1953 P. 0109 - 0110*\n*Danish special edition: Series I Chapter 1952-1958 P. 0009*\n*English special edition: Series I Chapter 1952-1958 P. 0009*\n*Greek special edition: Chapter 08 Volume 1 P. 0005*\n*Spanish special edition: Chapter 08 Volume 1 P. 0005*\n*Portuguese special edition Chapter 08 Volume 1 P. 0005*\n*Finnish special edition: Chapter 12 Volume 3 P. 0003*\n*Swedish special edition: Chapter 12 Volume 3 P. 0003*', meta=DocMeta(schema_name='docling_core.transforms.chunker.DocMeta', version='1.0.0', doc_items=[DocItem(self_ref='#/texts/1', parent=RefItem(cref='#/groups/0'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TEXT: 'text'>, prov=[], source=[], comments=[]), DocItem(self

### **Chunk Index Builder**

##### A. Helper Functions and Variables

In [16]:
# ---------------------------------------------------------------------
# Build FAISS chunk vector store
# ---------------------------------------------------------------------

def build_chunk_vector_store(
    processed_chunks,
    embedding_model,
    batch_size=64,
):
    """
    Build a FAISS vector store over document chunks.

    Parameters
    ----------
    processed_chunks : list[dict]

        Expected format:

        {
            "doc_id": str,
            "chunk_id": str,
            "chunk": str
        }

    embedding_model : SentenceTransformer
    batch_size : int

    Returns
    -------
    index : faiss.Index
        FAISS similarity index.

    chunk_lookup : dict
        Maps FAISS ids to chunk metadata.
    """

    if not processed_chunks:
        raise ValueError("processed_chunks is empty.")

    # -------------------------------------------------------------
    # Extract chunk texts
    # -------------------------------------------------------------

    texts = [
        chunk["chunk"].text
        for chunk in processed_chunks
    ]


    # -------------------------------------------------------------
    # Generate embeddings
    # -------------------------------------------------------------

    embeddings = embedding_model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    embeddings = np.asarray(embeddings, dtype=np.float32)


    # -------------------------------------------------------------
    # Build FAISS index
    #
    # Inner product on normalized vectors
    # = cosine similarity
    # -------------------------------------------------------------

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)


    # -------------------------------------------------------------
    # Metadata lookup
    # -------------------------------------------------------------

    chunk_lookup = dict()

    for idx, chunk in enumerate(processed_chunks):
        chunk_lookup[idx] = {
            "doc_id": chunk["doc_id"],
            "chunk_id": chunk["chunk_id"],
            "content": chunk["chunk"].text
        }

    return index, chunk_lookup


# ---------------------------------------------------------------------
# Query FAISS chunk store
# ---------------------------------------------------------------------

def search_chunk_vector_store(
    query,
    embedding_model,
    index,
    chunk_lookup,
    top_k=5,
):
    """
    Retrieve the most similar chunks.

    Parameters
    ----------
    query : str
    embedding_model : SentenceTransformer
    index : faiss.Index
    chunk_lookup : dict
    top_k : int

    Returns
    -------
    list[dict]
        Retrieved chunks with similarity scores.
    """


    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
    )

    query_embedding = np.asarray(query_embedding, dtype=np.float32)
    scores, indices = index.search(query_embedding, top_k)

    results = list()

    for score, idx in zip(scores[0], indices[0]):

        # FAISS returns -1 when no result exists
        if idx == -1:
            continue

        result = chunk_lookup[idx].copy()
        result["score"] = float(score)
        results.append(result)

    return results


# ---------------------------------------------------------------------
# Save / load FAISS index
# ---------------------------------------------------------------------

def save_chunk_vector_store(
    index,
    chunk_lookup,
    index_path,
    metadata_path,
):
    """
    Save FAISS index and metadata.
    """

    faiss.write_index(index, index_path)

    with open(metadata_path, "wb") as f:
        pickle.dump(chunk_lookup, f)


def load_chunk_vector_store(index_path, metadata_path):
    """
    Load FAISS index and metadata.
    """

    index = faiss.read_index(index_path)

    with open(metadata_path, "rb") as f:
        chunk_lookup = pickle.load(f)

    return index, chunk_lookup

##### B. Execution

In [17]:
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_index, chunk_lookup = build_chunk_vector_store(
    raw_chunks[:100],
    emb_model # From the Semantic Matching Module
)

INFO:sentence_transformers.base.model:No device provided, using cpu
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Re

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [18]:
# PICKLE THE VECTOR STORE!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

### **Chunk Processor**

##### A. Execution

In [19]:
nlp = spacy.load("en_core_web_lg")

def strip_all_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

# ---------------------------------------------------------------------
# Batch chunks
# ---------------------------------------------------------------------

docs = nlp.pipe(
    (item["chunk"].text for item in raw_chunks),
    batch_size=64,
    n_process=2
)

# ---------------------------------------------------------------------
# Process chunks
# ---------------------------------------------------------------------

processed_chunks = list()

for item, chunk in zip(raw_chunks, docs):

    filename = item["doc_id"]
    chunk_id = item["chunk_id"]

    filtered_sents = list()

    for sent in chunk.sents:
        text = sent.text.strip()

        if not text:
            continue

        # check for VERB or AUX in the sentence
        has_verb_or_aux = any(
            token.pos_ in {"VERB", "AUX"} for token in sent
        )

        if not has_verb_or_aux:
            continue

        filtered_sents.append(text)

    if not filtered_sents:
        continue

    processed_chunk = " ".join(filtered_sents)

    cleaned = strip_all_whitespace(processed_chunk)

    if cleaned:
        processed_chunks.append({
            "doc_id": filename,
            "chunk_id": chunk_id,
            "content": cleaned
        })

    print("CHUNK:", repr(cleaned))
    print("FILE:", filename, "\n")

CHUNK: 'No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel** *Official Journal 006 , 04/05/1953 P. 0109 - 0110* *Danish special edition: Series I Chapter 1952-1958 P. 0009* *English special edition: Series I Chapter 1952-1958 P. 0009* *Greek special edition: Chapter 08 Volume 1 P. 0005* *Spanish special edition: Chapter 08'
FILE: 31953D0030en.md 

CHUNK: "No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel THE HIGH AUTHORITY, Having regard to Article 60 and Article 63 (2) of the Treaty; Whereas compliance with the obligations of non-discrimination involves uniform application by undertakings of the conditions shown in their price lists with no other increases or reductions and no evasion of those obligations by allowing longer periods for settlement without a corresponding increase in price; Whereas the exception to this rule, namely the option of align

### **Assertion Extractor**

##### A. Helper Functions & Variables

In [20]:
# Security measure
_ = load_dotenv(find_dotenv()) # read local .env file

# For batch processing
SEMAPHORE = asyncio.Semaphore(30)

# Options:
# gpt-oss-120b:               OPENAI_API_KEY
# nemotron-3-super-120b-a12b: NEMOTRON_API_KEY
# nemotron-3-ultra-550b-a55b: NEMOTRON_ULTRA_API_KEY

openai.api_key  = os.getenv('NEMOTRON_ULTRA_API_KEY')

client = openai.AsyncOpenAI(
    base_url = "https://integrate.api.nvidia.com/v1",
    api_key = openai.api_key,
    max_retries=0
    )

extraction_schema = {
    "type": "object",
    "properties": {
        "assertions": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "assertion": {
                        "type": "object",
                        "properties": {
                            "subject": {
                                "type": "string"
                            },
                            "predicate": {
                                "type": "string"
                            },
                            "object": {
                                "oneOf": [
                                    {
                                        "type": "string"
                                    },
                                    {
                                        "type": "object",
                                        "properties": {
                                            "type": {
                                                "const": "assertion"
                                            }
                                        },
                                        "required": [
                                            "type"
                                        ],
                                        "additionalProperties": False
                                    }
                                ]
                            }
                        },
                        "required": [
                            "subject",
                            "predicate",
                            "object"
                        ],
                        "additionalProperties": False
                    },
                    "modifiers": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },
                "required": [
                    "assertion",
                    "modifiers"
                ],
                "additionalProperties": False
            }
        }
    },
    "required": [
        "assertions"
    ],
    "additionalProperties": False
}

extraction_prompt = """
Extract molecular legal assertions from the text.

Rules:

1. Extract each complete legal proposition as one assertion. Keep the
   subject, predicate, and object concise, but preserve all information
   necessary to identify the proposition's legal meaning.

2. Preserve the legal modality expressed by the text, such as shall, must,
   may, may not, shall not, is entitled to, is required to, or is prohibited
   from. Do not weaken, strengthen, or otherwise change the modality.

3. Keep the predicate limited to the legal relation and its modality.
   Do not absorb a prepositional phrase, infinitival complement, or clause
   into the predicate merely because it follows the verb.

4. Identify the semantic object of the predicate, rather than mechanically
   treating every phrase following the verb as part of the predicate or
   object.

5. Move only external conditions, exceptions, circumstances, temporal
   restrictions, purposes, or other qualifications of the complete
   proposition into "modifiers".

6. Do not move information into "modifiers" merely because it is long.
   Relative clauses, prepositional phrases, and other information necessary
   to identify the subject or object should remain attached to that entity.

7. Split coordinated or sequential clauses when they express independent
   legal propositions, especially when they contain separate legal
   modalities or obligations. Keep each proposition's complete predicate
   with its corresponding subject and object.

8. Coordinated subjects or objects should be represented so that they can be
   expanded safely downstream.

9. Resolve a reference only when its antecedent is explicitly identifiable
   from the provided text. Otherwise preserve the reference as written.
   Do not guess or invent an antecedent.

10. Do not invent an object merely to force an assertion into a
    subject-predicate-object structure. Some legal predicates are naturally
    objectless. For such propositions, set "object" to null.

11. Distinguish the semantic object of an assertion from external
    qualifications of that proposition.

12. A modifier may be long when the legal condition is genuinely complex.
    Do not shorten or discard legally meaningful details merely to satisfy
    a length preference.

13. Do not split a single coherent legal condition into multiple assertions
    unless it contains independent legal propositions.

14. Preserve explicit legal references such as Articles, paragraphs,
    sections, annexes, and other cited provisions.

15. Prefer conservative extraction over speculative interpretation.
    Extract the legal structure expressed by the text; do not rewrite a
    proposition into a different legal formulation merely to make it more
    concise or natural.

Structural conventions:

- The predicate should normally contain the modal/legal relation and its
  governing verb or predicate complement, e.g. "shall submit", "may approve",
  "shall not disclose", or "is entitled to receive".

- A noun phrase that identifies what is acted upon should normally be the
  object, even when that noun phrase is long.

- Relative clauses and other phrases that identify or qualify an entity
  should remain attached to the subject or object when they are part of
  that entity's description.

- Conditions, exceptions, temporal restrictions, purposes, and
  circumstances governing when or under what circumstances the proposition
  applies should normally be represented as modifiers.

- Do not treat an objectless passive construction as having an invented
  object. For example, "The register shall be maintained" has object = null.

- Do not convert legal framing into a different modality. For example,
  do not transform "is prohibited from" into "shall not", or "may" into
  "is entitled to", unless the text itself expresses that relation.

- When a verb governs an infinitival or clausal complement that expresses
  the action required, permitted, or prohibited, keep the legal relation
  coherent rather than splitting the complement into an unrelated
  predicate.

Examples:

Input:
"Any undertaking established within the territory must maintain records
relating to goods supplied to its customers."

Output:
{
  "assertion": {
    "subject": "undertaking established within the territory",
    "predicate": "must maintain",
    "object": "records relating to goods supplied to its customers"
  },
  "modifiers": []
}

---

Input:
"Where the authority considers that the application is incomplete,
the applicant must provide the missing information within thirty days."

Output:
{
  "assertion": {
    "subject": "applicant",
    "predicate": "must provide",
    "object": "missing information"
  },
  "modifiers": [
    "Where the authority considers that the application is incomplete",
    "within thirty days"
  ]
}

---

Input:
"The authority shall inform the applicant of the reasons for its
decision and shall provide a copy of the decision."

Output:
{
  "assertions": [
    {
      "assertion": {
        "subject": "authority",
        "predicate": "shall inform",
        "object": "applicant of the reasons for its decision"
      },
      "modifiers": []
    },
    {
      "assertion": {
        "subject": "authority",
        "predicate": "shall provide",
        "object": "a copy of the decision"
      },
      "modifiers": []
    }
  ]
}

---

Input:
"The authorisation may be withdrawn if the holder fails to comply
with the applicable requirements."

Output:
{
  "assertion": {
    "subject": "authorisation",
    "predicate": "may be withdrawn",
    "object": null
  },
  "modifiers": [
    "if the holder fails to comply with the applicable requirements"
  ]
}

---

Input:
"The authority shall verify the documents submitted by applicants
and may reject applications that do not satisfy the requirements."

Output:
{
  "assertions": [
    {
      "assertion": {
        "subject": "authority",
        "predicate": "shall verify",
        "object": "documents submitted by applicants"
      },
      "modifiers": []
    },
    {
      "assertion": {
        "subject": "authority",
        "predicate": "may reject",
        "object": "applications that do not satisfy the requirements"
      },
      "modifiers": []
    }
  ]
}

---

Text:
"""

async def get_completion(
        system_prompt="",
        query="",
        schema=extraction_schema
        ):
    
    completion = await client.chat.completions.create(
        # Model Options:
        # openai/gpt-oss-120b
        # nvidia/nemotron-3-nano-30b-a3b
        # nvidia/nemotron-3-nano-omni-30b-a3b-reasoning
        # nvidia/nemotron-3.5-lightning-30b-a3b
        # nvidia/nemotron-3-super-120b-a12b
        # nvidia/nemotron-3-ultra-550b-a55b

        model="nvidia/nemotron-3-ultra-550b-a55b",
        messages=[
           {'role':'system', 'content': system_prompt},
           {'role':'user', 'content': query}
           ],
        temperature=0,
        top_p=0.000001, # Test different values
        max_tokens=6000,
        stream=False,
        reasoning_effort="none",
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "triples",
                "strict": True,
                "schema": schema
                }
            }
        )

    return completion.choices[0].message.content


# ---------------------------------------------------------------------
# Global rate limiter
# ---------------------------------------------------------------------

async def wait_for_rate_limit(
    rate_lock,
    last_request_time,
    cooldown_until,
    current_rps,
):

    async with rate_lock:
        now = time.monotonic()

        # ----------------------------------------------------------
        # Global cooldown
        # ----------------------------------------------------------

        if now < cooldown_until[0]:
            await asyncio.sleep(cooldown_until[0] - now)
            now = time.monotonic()

        # ----------------------------------------------------------
        # Request spacing
        # ----------------------------------------------------------

        delay = 1 / current_rps[0]
        elapsed = (now - last_request_time[0])
        wait_time = delay - elapsed

        if wait_time > 0:
            await asyncio.sleep(wait_time)
            now = time.monotonic()

        last_request_time[0] = now


# ---------------------------------------------------------------------
# Extraction worker
# ---------------------------------------------------------------------

async def extraction_worker(
    chunk_dict,
    semaphore,
    rate_lock,
    last_request_time,
    cooldown_until,
    current_rps,
    success_counter,
    max_rps,
    max_backoff=30,
    min_rps=0.25,
    recovery_step=0.10,
    success_threshold=20,
):

    attempt = 0

    while True:
        async with semaphore:
            await wait_for_rate_limit(
                rate_lock,
                last_request_time,
                cooldown_until,
                current_rps,
            )

            try:
                response = await get_completion(
                    system_prompt=extraction_prompt,
                    query=chunk_dict["content"],
                    schema=extraction_schema,
                )

                result = json.loads(response)

                # ------------------------------------------
                # Conservative recovery
                # ------------------------------------------

                async with rate_lock:
                    success_counter[0] += 1

                    if success_counter[0] >= success_threshold:
                        current_rps[0] = min(max_rps, current_rps[0] + recovery_step,)
                        success_counter[0] = 0

                return {
                    "doc_id": chunk_dict["doc_id"],
                    "chunk_id": chunk_dict["chunk_id"],
                    "triples": result,
                }

            except Exception as e:
                error_text = str(e)

                # ------------------------------------------
                # Retry indefinitely for transient errors
                # ------------------------------------------
                if (
                    "429" in error_text
                    or
                    "503" in error_text
                ):

                    backoff = min(2 ** attempt, max_backoff)
                    backoff += random.uniform(0, 1)

                    async with rate_lock:
                        cooldown_until[0] = max(cooldown_until[0], time.monotonic() + backoff)
                        current_rps[0] = max(min_rps, current_rps[0] / 2)
                        success_counter[0] = 0
                        current_rate = current_rps[0]

                    print(
                        f"[{chunk_dict['chunk_id']}] "
                        f"{error_text}\n"
                        f"Cooldown: {backoff:.1f}s | "
                        f"Current RPS: {current_rate:.2f}"
                    )

                    attempt += 1

                    continue

                # ------------------------------------------
                # Permanent failure
                # ------------------------------------------

                print(
                    f"Extraction failed "
                    f"({chunk_dict['chunk_id']}): {e}"
                )

                return {
                    "doc_id": chunk_dict["doc_id"],
                    "chunk_id": chunk_dict["chunk_id"],
                    "triples": list(),
                }


# ---------------------------------------------------------------------
# Batch extraction
# ---------------------------------------------------------------------

async def batch_extract(
    chunks,
    semaphore,
    requests_per_second=1,
    max_backoff=30,
):

    rate_lock = asyncio.Lock()

    last_request_time = [0.0]
    cooldown_until = [0.0]
    current_rps = [requests_per_second]
    success_counter = [0]

    tasks = [

        extraction_worker(
            chunk,
            semaphore,
            rate_lock,
            last_request_time,
            cooldown_until,
            current_rps,
            success_counter,
            requests_per_second,
            max_backoff,
        )

        for chunk in chunks

    ]

    return await asyncio.gather(*tasks)

##### B. Execution

In [62]:
# FOR PROMPT TUNING
async def test_run(
        system_prompt="",
        query="",
        ):
    
    completion = await client.chat.completions.create(
        # Model Options:
        # openai/gpt-oss-120b
        # nvidia/nemotron-3-nano-30b-a3b
        # nvidia/nemotron-3-super-120b-a12b
        # nvidia/nemotron-3-ultra-550b-a55b
        # nvidia/llama-3.3-nemotron-super-49b-v1.5
        # z-ai/glm-5.2

        model="nvidia/nemotron-3-nano-30b-a3b",
        messages=[
           {'role':'system', 'content': system_prompt},
           {'role':'user', 'content': query}
           ],
        temperature=0,
        top_p=0.000001, # Test different values
        max_tokens=6000,
        stream=False,
        reasoning_effort="none",
        )

    return completion.choices[0].message.content

In [21]:
await test_run(system_prompt="", query="Hello there!")

INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"


'Hello! How can I help you today?'

In [100]:
extraction_results = await batch_extract(processed_chunks[:30], SEMAPHORE, requests_per_second=2)

INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 503 Service Unavailable"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 503 Service Unavailable"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidi

[chunk_1] Error code: 503 - {'error': {'message': 'Internal server error', 'type': 'Service Unavailable', 'code': 503}}
Cooldown: 1.9s | Current RPS: 1.00
[chunk_2] Error code: 503 - {'error': {'message': 'Internal server error', 'type': 'Service Unavailable', 'code': 503}}
Cooldown: 1.8s | Current RPS: 0.50


INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 503 Service Unavailable"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"


[chunk_4] Error code: 503 - {'error': {'message': 'Internal server error', 'type': 'Service Unavailable', 'code': 503}}
Cooldown: 1.9s | Current RPS: 0.25


INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 503 Service Unavailable"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"


[chunk_2] Error code: 503 - {'error': {'message': 'Internal server error', 'type': 'Service Unavailable', 'code': 503}}
Cooldown: 2.3s | Current RPS: 0.25


INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.

In [97]:
extraction_results[1]

{'doc_id': '31953D0030en.md',
 'chunk_id': 'chunk_2',
 'triples': {'assertions': [{'assertion': {'subject': 'This Decision',
     'predicate': 'shall apply to',
     'object': 'Community undertakings in respect of their transactions within the common market in the products specified in Annex I to the Treaty, with the exception of scrap'},
    'modifiers': []},
   {'assertion': {'subject': 'seller',
     'predicate': 'shall be a prohibited practice within the meaning of Article 60 (1) of the Treaty to apply',
     'object': 'increases or reductions on the terms calculable, for the transaction concerned, for his published price list and conditions of sale'},
    'modifiers': []},
   {'assertion': {'subject': 'This Article',
     'predicate': 'shall be no bar to',
     'object': 'the application of Article 60 (2) (b) of the Treaty or of Article 4 below, nor to the granting by undertakings in the coal industry of quantity or loyalty bonuses not shown in price lists pursuant to Article 2 (3

In [25]:
processed_chunks[4]["content"]

'No 26-54 of 6 May 1954 laying down in implementation of Article 66 (4) of the Treaty a regulation concerning information to be furnished** *Official Journal 009 , 11/05/1954 P. 0350 - 0351* *Danish special edition: Series I Chapter 1952-1958 P. 0017* *English special edition: Series I Chapter 1952-1958 P. 0017* *Greek special edition: Chapter 08 Volume 1 P. 0014* *Spanish special edition: Chapter 08 Volume 1 P. 0015* *Portuguese special edition Chapter 08 Volume 1 P. 0015* *Finnish special edition: Chapter 8 Volume 1 P. 0004* *Swedish special edition: Chapter 8 Volume 1 P. 0004*'

In [101]:
import re
from collections import Counter


# Only genuine anaphoric pronouns.
# Deliberately excludes: this, that, these, those, which, who, whom, whose.
PRONOUNS = {
    "it", "its",
    "they", "them", "their", "theirs",
    "he", "him", "his",
    "she", "her", "hers",
}


SUSPICIOUS_PREDICATE_PATTERNS = [
    r"\bshall\s+replaced\b",
    r"\bshall\s+replaced\s+for\b",
    r"\bmay\s+be\s+of\b",
]


def diagnose_assertions(
    result,
    predicate_words=15,
    modifier_words=45,
):
    """
    Lightweight diagnostic for extracted assertions.

    This is a candidate generator, NOT a semantic evaluator.
    It flags assertions that are worth manual inspection.

    Primary flags:
        - empty_object
        - empty_object_with_modifier
        - suspicious_predicate
        - long_predicate
        - long_modifier

    Secondary flag:
        - possible_coreference

    Long subjects/objects are intentionally NOT treated as
    primary errors because long legal components may be faithful.
    """

    assertions = result["triples"]["assertions"]

    flagged = []

    for i, item in enumerate(assertions):

        assertion = item.get("assertion", {})
        modifiers = item.get("modifiers", [])

        subject = str(assertion.get("subject", "")).strip()
        predicate = str(assertion.get("predicate", "")).strip()
        obj = str(assertion.get("object", "")).strip()

        flags = []

        # ---------------------------------------------------------
        # 1. Empty object
        # ---------------------------------------------------------

        if not obj:
            flags.append("empty_object")

            if modifiers:
                flags.append("empty_object_with_modifier")

        # ---------------------------------------------------------
        # 2. Long predicate
        # ---------------------------------------------------------

        if len(predicate.split()) > predicate_words:
            flags.append("long_predicate")

        # ---------------------------------------------------------
        # 3. Long modifiers
        # ---------------------------------------------------------

        for modifier in modifiers:
            if len(str(modifier).split()) > modifier_words:
                flags.append("long_modifier")
                break

        # ---------------------------------------------------------
        # 4. Possible unresolved coreference
        # ---------------------------------------------------------

        core_text = f"{subject} {predicate} {obj}".lower()

        words = set(
            re.findall(r"\b[\w'-]+\b", core_text)
        )

        if words & PRONOUNS:
            flags.append("possible_coreference")

        # ---------------------------------------------------------
        # 5. Suspicious predicate wording
        # ---------------------------------------------------------

        predicate_lower = predicate.lower()

        for pattern in SUSPICIOUS_PREDICATE_PATTERNS:
            if re.search(pattern, predicate_lower):
                flags.append("suspicious_predicate")
                break

        # ---------------------------------------------------------
        # Store candidate
        # ---------------------------------------------------------

        if flags:
            flagged.append({
                "index": i,
                "flags": flags,
                "assertion": assertion,
                "modifiers": modifiers,
            })

    # -------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------

    flag_counts = Counter(
        flag
        for item in flagged
        for flag in item["flags"]
    )

    print(f"Total assertions: {len(assertions)}")
    print(f"Flagged assertions: {len(flagged)}")
    print()

    print("Flag counts:")

    if flag_counts:
        for flag, count in flag_counts.most_common():
            print(f"  {flag}: {count}")
    else:
        print("  None")

    # -------------------------------------------------------------
    # Print only the high-priority candidates
    # -------------------------------------------------------------

    HIGH_PRIORITY = {
        "empty_object",
        "empty_object_with_modifier",
        "long_predicate",
        "long_modifier",
        "suspicious_predicate",
    }

    high_priority = [
        item for item in flagged
        if set(item["flags"]) & HIGH_PRIORITY
    ]

    print("\n" + "=" * 80)
    print("HIGH-PRIORITY ASSERTIONS")
    print("=" * 80)

    if not high_priority:
        print("None")

    for item in high_priority:

        print(
            f"\n[{item['index']}] "
            f"{', '.join(item['flags'])}"
        )

        print(
            f"  S: {item['assertion'].get('subject', '')}"
        )

        print(
            f"  P: {item['assertion'].get('predicate', '')}"
        )

        print(
            f"  O: {item['assertion'].get('object', '')}"
        )

        if item["modifiers"]:
            print("  Modifiers:")

            for modifier in item["modifiers"]:
                print(f"    - {modifier}")

    return {
        "total": len(assertions),
        "flagged": flagged,
        "high_priority": high_priority,
        "flag_counts": dict(flag_counts),
    }

diagnoses = []

for result in extraction_results:
    print("\n" + "#" * 80)
    print(f"{result['doc_id']} / {result['chunk_id']}")
    print("#" * 80)

    diagnosis = diagnose_assertions(result)
    diagnoses.append(diagnosis)


################################################################################
31953D0030en.md / chunk_1
################################################################################
Total assertions: 0
Flagged assertions: 0

Flag counts:
  None

HIGH-PRIORITY ASSERTIONS
None

################################################################################
31953D0030en.md / chunk_2
################################################################################
Total assertions: 8
Flagged assertions: 4

Flag counts:
  possible_coreference: 4

HIGH-PRIORITY ASSERTIONS
None

################################################################################
31953D0030en.md / chunk_3
################################################################################
Total assertions: 4
Flagged assertions: 2

Flag counts:
  possible_coreference: 2

HIGH-PRIORITY ASSERTIONS
None

################################################################################
31954S0024en.md / chunk_1
#####

In [ ]:
extraction_results = await batch_extract(
    processed_chunks,
    SEMAPHORE,
    requests_per_second=2,
)

extracted_assertions = list()
extraction_errors = list()

for result in extraction_results:

    doc_id = result.get("doc_id")
    chunk_id = result.get("chunk_id")
    assertions = result.get("assertions")

    # -----------------------------------------------------------------
    # Validate extraction output
    # -----------------------------------------------------------------

    if assertions is None:
        extraction_errors.append(
            {
                "doc_id": doc_id,
                "chunk_id": chunk_id,
                "reason": "missing_assertions",
                "raw_result": result,
            }
        )
        continue

    if not isinstance(assertions, list):

        extraction_errors.append(
            {
                "doc_id": doc_id,
                "chunk_id": chunk_id,
                "reason": (
                    f"invalid_assertions_type:"
                    f"{type(assertions).__name__}"
                ),
                "raw_result": result,
            }
        )

        continue

    # -----------------------------------------------------------------
    # Keep valid extraction result
    # -----------------------------------------------------------------

    extracted_assertions.append(
        {
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "assertions": assertions,
        }
    )


# ---------------------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------------------

print("=" * 80)
print("ASSERTION EXTRACTION RESULTS")
print("=" * 80)

print(
    f"Extraction results : "
    f"{len(extraction_results):,}"
)

print(
    f"Valid chunks       : "
    f"{len(extracted_assertions):,}"
)

print(
    f"Errors             : "
    f"{len(extraction_errors):,}"
)

total_assertions = sum(
    len(item["assertions"])
    for item in extracted_assertions
)

print(
    f"Extracted assertions: "
    f"{total_assertions:,}"
)

if extraction_errors:

    print()
    print("ERRORS")
    print("-" * 80)

    for error in extraction_errors:
        print(
            f"{error['doc_id']} / "
            f"{error['chunk_id']} "
            f"→ {error['reason']}"
        )

INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 503 Service Unavailable"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/com

[chunk_2] Error code: 503 - {'error': {'message': 'ResourceExhausted: Worker local total request limit reached (32/32)', 'type': 'Service Unavailable', 'code': 503}}
Cooldown: 1.4s | Current RPS: 1.00
[chunk_2] Error code: 503 - {'error': {'message': 'ResourceExhausted: Worker local total request limit reached (33/32)', 'type': 'Service Unavailable', 'code': 503}}
Cooldown: 1.8s | Current RPS: 0.50
[chunk_6] Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Cooldown: 1.7s | Current RPS: 0.25


INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/compl

[chunk_7] Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Cooldown: 1.5s | Current RPS: 0.25
[chunk_9] Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Cooldown: 1.8s | Current RPS: 0.25
[chunk_10] Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Cooldown: 1.5s | Current RPS: 0.25
[chunk_12] Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Cooldown: 1.3s | Current RPS: 0.25
[chunk_13] Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Cooldown: 1.9s | Current RPS: 0.25
[chunk_1] Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Cooldown: 1.7s | Current RPS: 0.25
[chunk_1] Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Cooldown: 1.2s | Current RPS: 0.25
[chunk_1] Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Cooldown: 1.0s | Current RPS: 0.25
[chunk_1] Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Cooldown: 1.6s | Current RPS: 0.25


INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.

[chunk_7] Error code: 503 - {'error': {'message': 'ResourceExhausted: Worker local total request limit reached (32/32)', 'type': 'Service Unavailable', 'code': 503}}
Cooldown: 1.6s | Current RPS: 0.25
[chunk_3] Error code: 503 - {'error': {'message': 'ResourceExhausted: Worker local total request limit reached (96/32)', 'type': 'Service Unavailable', 'code': 503}}
Cooldown: 1.6s | Current RPS: 0.25
[chunk_2] Error code: 503 - {'error': {'message': 'ResourceExhausted: Worker local total request limit reached (33/32)', 'type': 'Service Unavailable', 'code': 503}}
Cooldown: 1.7s | Current RPS: 0.25
[chunk_7] Error code: 503 - {'error': {'message': 'ResourceExhausted: Worker local total request limit reached (32/32)', 'type': 'Service Unavailable', 'code': 503}}
Cooldown: 1.5s | Current RPS: 0.25


INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.

[chunk_1] Error code: 503 - {'error': {'message': 'ResourceExhausted: Worker local total request limit reached (104/32)', 'type': 'Service Unavailable', 'code': 503}}
Cooldown: 2.8s | Current RPS: 0.25


INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.

### **Assertion Normalizer**

##### A. Helper Functions & Variables

In [ ]:
nlp = spacy.load(
    "en_core_web_sm",
    disable=["ner"]
)

# ---------------------------------------------------------------------
# Normalization
# ---------------------------------------------------------------------

def normalize_entity(text: str) -> str:
    """
    Conservative cleanup of entity names.
    Preserves legal wording.
    """

    text = text.strip()

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove trailing punctuation
    text = text.rstrip(".,;:")

    return text


def normalize_predicate(text: str) -> str:
    """
    Cleanup and canonicalize predicate phrases.
    """

    text = re.sub(r"\s+", " ", text.strip())

    replacements = {
        "is obliged to": "obligated to",
        "is required to": "required to",
        "is subject to": "subject to",
        "is entitled to": "entitled to",
    }

    lower = text.lower()

    for old, new in replacements.items():
        if lower.startswith(old):
            text = new + text[len(old):]
            break

    return text


# ---------------------------------------------------------------------
# Legal reference splitting
# ---------------------------------------------------------------------

def split_legal_reference(text: str):

    pattern = (
        r"\s+(?:and|or)\s+"
        r"(?=(?:article|paragraph|point|section|annex)\b)"
    )

    parts = re.split(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    if len(parts) > 1:
        return [
            part.strip()
            for part in parts
        ]

    return None


# ---------------------------------------------------------------------
# Cheap composite detection heuristic
# ---------------------------------------------------------------------

def should_attempt_split(text: str):
    """
    Avoid unnecessary spaCy parsing.

    Only attempt NLP splitting when the text
    contains likely coordination markers.
    """

    indicators = [
        " and ",
        " or ",
        ",",
        "Article ",
        "Paragraph ",
        "Point ",
        "Section ",
        "Annex "
    ]

    text_lower = text.lower()

    return any(
        indicator.lower() in text_lower
        for indicator in indicators
    )


# ---------------------------------------------------------------------
# Composite entity detection
# ---------------------------------------------------------------------

def split_entity(text: str):
    """
    Returns:
        None              -> no safe expansion
        list[str]         -> safe expansion
    """

    # First handle explicit legal references
    legal_split = split_legal_reference(text)

    if legal_split:
        return legal_split


    # Skip expensive NLP when unlikely to be composite
    if not should_attempt_split(text):
        return None


    # Fall back to dependency parsing
    doc = nlp(text)


    # Require coordination
    if not any(
        tok.dep_ == "conj"
        for tok in doc
    ):
        return None


    # Avoid complex clauses
    if any(
        tok.dep_ in {
            "relcl",
            "advcl",
            "ccomp",
            "xcomp"
        }
        for tok in doc
    ):
        return None


    noun_chunks = list(doc.noun_chunks)

    if len(noun_chunks) < 2:
        return None

    entities = [
        chunk.text.strip()
        for chunk in noun_chunks
    ]


    entities = list(dict.fromkeys(entities))

    return (
        entities
        if len(entities) > 1
        else None
    )


# ---------------------------------------------------------------------
# Assertion expansion
# ---------------------------------------------------------------------

def expand_assertion(assertion: dict):

    subject = normalize_entity(assertion["subject"])
    predicate = normalize_predicate(assertion["predicate"])
    object = normalize_entity(assertion["object"])

    subjects = (
        split_entity(subject)
        or [subject]
    )

    objects = (
        split_entity(object)
        or [object]
    )

    expanded = list()

    if (
        len(subjects) == len(objects)
        and len(subjects) > 1
    ):
        pairs = zip(subjects, objects)

    else:
        pairs = product(subjects, objects)

    for s, o in pairs:
        expanded.append(
            {
                "subject": s,
                "predicate": predicate,
                "object": o,
            }
        )

    return expanded


# ---------------------------------------------------------------------
# Batch normalization
# ---------------------------------------------------------------------

def normalize_assertions(extracted_assertions):

    normalized = list()

    required_fields = {
        "subject",
        "predicate",
        "object",
    }

    for chunk in extracted_assertions:

        doc_id = chunk["doc_id"]
        chunk_id = chunk["chunk_id"]

        assertion = chunk.get("assertion")
        modifiers = chunk.get("modifiers", [])

        if not isinstance(assertion, dict):
            continue

        if not required_fields.issubset(assertion):
            continue

        expanded = expand_assertion(assertion)

        for normalized_assertion in expanded:

            normalized.append(
                {
                    "doc_id": doc_id,
                    "chunk_id": chunk_id,
                    "assertion_id": len(normalized),
                    "assertion": normalized_assertion,
                    "modifiers": modifiers.copy(),
                }
            )

    return normalized

##### B. Execution

In [ ]:
normalized_assertions = normalize_assertions(extracted_assertions)

### **Semantic Matcher**

##### A. Helper Functions & Variables

In [ ]:
TYPE_MAP = {
    "string": XSD.string,
    "integer": XSD.integer,
    "decimal": XSD.decimal,
    "boolean": XSD.boolean,
    "date": XSD.date,
    "datetime": XSD.dateTime,
}

DATE_FORMATS = [
    "%Y-%m-%d",

    # Common EU/legal formats
    "%d-%m-%Y",
    "%d/%m/%Y",
    "%d.%m.%Y",

    # Written dates
    "%d %B %Y",
    "%d %b %Y",
]

DATETIME_FORMATS = [
    "%Y-%m-%dT%H:%M:%S",
    "%Y-%m-%dT%H:%M:%SZ",
    "%Y-%m-%dT%H:%M:%S%z",

    # ISO with fractional seconds
    "%Y-%m-%dT%H:%M:%S.%f",
    "%Y-%m-%dT%H:%M:%S.%fZ",
    "%Y-%m-%dT%H:%M:%S.%f%z",
]

LEGAL_IDENTIFIER_PATTERNS = [

    # Article / paragraph references
    r"\b(article|art\.?)\s*\d+(\s*\([a-z0-9]+\))?",

    # EU legal acts with numbers
    # Regulation No 1395/69
    # Decision 68/302/EEC
    # Directive 2006/123/EC
    r"\b(regulation|directive|decision|recommendation|opinion)"
    r".*\b(no\.?|number)?\s*\d+[-/]\d+",

    # EU identifiers:
    # 2006/123/EC
    # 68/302/EEC
    r"\b\d{1,4}/\d{1,4}/[a-z]{2,5}\b",

    # CELEX-like identifiers
    r"\b\d{5}[A-Z]\d{4}\b",
]

NEGATION_WORDS = {
    "not",
    "no",
    "never",
    "without",
    "neither",
    "nor",
    "n't"
}

def detect_literal_type(value: str):

    v = clean_text(str(value)).strip()

    # boolean
    if v.lower() in {"true", "false"}:
        return "boolean"

    # integer
    if re.fullmatch(r"-?\d+", v):
        return "integer"

    # decimal
    if re.fullmatch(r"-?\d+\.\d+", v):
        return "decimal"

    # datetime
    for fmt in DATETIME_FORMATS:
        try:
            datetime.strptime(v, fmt)
            return "datetime"
        except ValueError:
            pass

    # date
    for fmt in DATE_FORMATS:
        try:
            datetime.strptime(v, fmt)
            return "date"
        except ValueError:
            pass

    return "class"

def normalize_date(value: str, literal_type: str):
    # value is expected to already be cleaned by the caller
    if literal_type == "date":
        for fmt in DATE_FORMATS:
            try:
                dt = datetime.strptime(value, fmt)
                return dt.strftime("%Y-%m-%d")
            except ValueError:
                continue

    elif literal_type == "datetime":
        for fmt in DATETIME_FORMATS:
            try:
                dt = datetime.strptime(value, fmt)
                return dt.isoformat()
            except ValueError:
                continue

    return value

def is_legal_identifier_reference(text: str) -> bool:
    """
    Detect whether a label contains a likely legal identifier.

    This is a heuristic filter only.
    """

    text = text.casefold()

    for pattern in LEGAL_IDENTIFIER_PATTERNS:
        if re.search(pattern, text):
            return True

    return False

def extract_polarity_doc(doc):
    """
    Returns True for affirmative predicates,
    False for explicitly negative predicates.

    Designed for short ontology labels rather than
    full natural-language sentences.
    """

    for token in doc:

        if token.lower_ in NEGATION_WORDS:
            return False

    return True

def extract_polarity(text):
    """
    Convenience wrapper when a Doc is unavailable.
    """

    return extract_polarity_doc(
        nlp.make_doc(text)
    )

def create_custom_uri(text: str, prefix=JS_DATA):
    """
    Generate a stable URI for unresolved components.

    Applies conservative normalization only.
    """

    text = text.casefold()

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Replace whitespace with underscores
    text = re.sub(r"\s+", "_", text)

    # Remove unsafe URI characters
    text = re.sub(r"[^a-z0-9_\-]", "", text)

    return URIRef(prefix[text])


def build_resource_indices(
    resources,
    resource_metadata,
    emb_model,
):
    """
    Build FAISS indices for ontology resources.

    Returns:
        {
            resource_type: (index, uris, labels)
        }
    """

    index_lookup = dict()

    for resource_type, resource_dict in resources.items():

        uris = list(resource_dict.keys())

        texts = [
            resource_metadata[uri]["text"]
            for uri in uris
        ]

        embeddings = emb_model.encode(
            texts,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

        index = faiss.IndexFlatIP(
            embeddings.shape[1]
        )

        index.add(embeddings)

        index_lookup[resource_type] = (
            index,
            uris,
            texts,
        )

    return index_lookup


def collect_lookup_requests(
    normalized_assertions,
):
    """
    Prepare assertions for semantic resource matching.

    Modifiers are preserved unchanged.
    """

    lookup_requests = defaultdict(list)

    scored_assertions = copy.deepcopy(
        normalized_assertions
    )

    for idx, record in enumerate(
        normalized_assertions
    ):

        assertion = record["assertion"]
        scored_assertion = (
            scored_assertions[idx]["assertion"]
        )

        # --------------------------------------------------
        # Subject
        # --------------------------------------------------

        if is_legal_identifier_reference(
            assertion["subject"]
        ):
            scored_assertion["subject"] = (
                create_custom_uri(
                    assertion["subject"]
                )
            )

        else:
            lookup_requests["class"].append(
                {
                    "idx": idx,
                    "field": "subject",
                    "text": str(
                        assertion["subject"]
                    ),
                }
            )

        # --------------------------------------------------
        # Object
        # --------------------------------------------------

        object_type = detect_literal_type(
            assertion["object"]
        )

        if object_type == "class":

            if is_legal_identifier_reference(
                assertion["object"]
            ):
                scored_assertion["object"] = (
                    create_custom_uri(
                        assertion["object"]
                    )
                )

            else:
                lookup_requests["class"].append(
                    {
                        "idx": idx,
                        "field": "object",
                        "text": str(
                            assertion["object"]
                        ),
                    }
                )

        else:

            clean_value = (
                clean_text(
                    str(assertion["object"])
                ).strip()
            )

            if object_type in {
                "date",
                "datetime",
            }:
                value = normalize_date(
                    clean_value,
                    object_type,
                )

            elif object_type == "integer":
                value = int(clean_value)

            elif object_type == "decimal":
                value = float(clean_value)

            else:
                value = scored_assertion["object"]

            literal = Literal(
                value,
                datatype=TYPE_MAP[object_type],
            )

            if object_type == "date":
                try:
                    rdflib.xsd_datetime.parse_xsd_date(
                        str(literal)
                    )

                except Exception as exc:
                    raise ValueError(
                        f"Bad date literal at idx={idx}: "
                        f"raw={assertion['object']!r} "
                        f"clean={clean_value!r} "
                        f"normalized={value!r}"
                    ) from exc

            scored_assertion["object"] = literal

        # --------------------------------------------------
        # Predicate
        # --------------------------------------------------

        if assertion["predicate"] == "type of":

            scored_assertion["predicate"] = RDF.type

        else:

            property_type = (
                "obj_prop"
                if object_type == "class"
                else "datatype_prop"
            )

            lookup_requests[
                property_type
            ].append(
                {
                    "idx": idx,
                    "field": "predicate",
                    "text": str(
                        assertion["predicate"]
                    ),
                }
            )

    return (
        scored_assertions,
        lookup_requests,
    )


def perform_semantic_lookups(
    lookup_requests,
    index_lookup,
    emb_model,
    top_k=3,
    batch_size=128,
):
    """
    Perform batched semantic searches against
    the ontology resource indices.
    """

    lookup_results = dict()
    polarity_cache = dict()

    for resource_type, requests in (
        lookup_requests.items()
    ):

        if not requests:
            continue

        texts = [
            request["text"]
            for request in requests
        ]

        embeddings = emb_model.encode(
            texts,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

        index, uris, labels = (
            index_lookup[resource_type]
        )

        scores, indices = index.search(
            embeddings,
            top_k,
        )

        for request, score_row, idx_row in zip(
            requests,
            scores,
            indices,
        ):

            candidates = [
                (
                    labels[i],
                    uris[i],
                    float(score_row[j]),
                )
                for j, i in enumerate(idx_row)
                if i >= 0
            ]

            # --------------------------------------------------
            # Predicate polarity filtering
            # --------------------------------------------------

            if request["field"] == "predicate":

                query_text = request["text"]

                if query_text not in polarity_cache:
                    polarity_cache[query_text] = (
                        extract_polarity(
                            query_text
                        )
                    )

                query_polarity = (
                    polarity_cache[query_text]
                )

                filtered_candidates = []

                for candidate in candidates:

                    candidate_label = candidate[0]

                    if (
                        candidate_label
                        not in polarity_cache
                    ):
                        polarity_cache[
                            candidate_label
                        ] = extract_polarity(
                            candidate_label
                        )

                    if (
                        polarity_cache[
                            candidate_label
                        ]
                        == query_polarity
                    ):
                        filtered_candidates.append(
                            candidate
                        )

                if filtered_candidates:
                    candidates = (
                        filtered_candidates
                    )

            lookup_results[
                (
                    request["idx"],
                    request["field"],
                    request["text"],
                )
            ] = candidates

    return lookup_results


def apply_resource_matches(
    scored_assertions,
    lookup_results,
    threshold=0.7,
):
    """
    Replace unresolved assertion components with either
    matched ontology URIs or Jurisynth custom URIs.

    Modifiers remain unchanged.
    """

    for (
        idx,
        field,
        text,
    ), candidates in lookup_results.items():

        scored_assertion = (
            scored_assertions[idx]["assertion"]
        )

        if not candidates:
            # No semantic candidate found.
            # Preserve the original text as a custom URI.
            scored_assertion[field] = (
                create_custom_uri(text)
            )
            continue

        best_label, best_uri, best_score = (
            candidates[0]
        )

        if best_score >= threshold:

            scored_assertion[field] = URIRef(
                best_uri
            )

        else:

            scored_assertion[field] = (
                create_custom_uri(text)
            )

    return scored_assertions

##### B. Execution

In [ ]:
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

resources = {
    "class": classes,
    "obj_prop": obj_properties,
    "datatype_prop": datatype_properties,
    "datatype": datatypes,
}

index_lookup = build_resource_indices(
    resources,
    resource_metadata,
    emb_model,
)

scored_assertions, lookup_requests = collect_lookup_requests(normalized_assertions)

lookup_results = perform_semantic_lookups(
    lookup_requests,
    index_lookup,
    emb_model,
    top_k=3,
    batch_size=128,
)

scored_assertions = apply_resource_matches(
    scored_assertions,
    lookup_results,
    threshold=0.7,
)

Batches:   0%|          | 0/72 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

##### C. Inspection

In [40]:
full_list = [
    (component_name)
    for element in scored_triples
    for component_name in element["triple"]
    ]

inspection_list = [
    (component_name, component_value, element["triple"])
    for element in scored_triples
    for component_name, component_value in element["triple"].items()
    if str(component_value).startswith("http://publications")
]

In [41]:
print(len(full_list), len(inspection_list))

14442 734


In [42]:
scored_triples[50:100]

[{'doc_id': '31953D0030en.md',
  'chunk_id': 'chunk_2',
  'triple_id': 50,
  'triple': {'subject': rdflib.term.URIRef('http://jurisynth/data/conditions_of_sale'),
   'predicate': rdflib.term.URIRef('http://jurisynth/data/are_prohibited_from_being_differentiated'),
   'object': rdflib.term.URIRef('http://jurisynth/data/such_purchasers')}},
 {'doc_id': '31953D0030en.md',
  'chunk_id': 'chunk_2',
  'triple_id': 51,
  'triple': {'subject': rdflib.term.URIRef('http://jurisynth/data/conditions_of_sale'),
   'predicate': rdflib.term.URIRef('http://jurisynth/data/are_prohibited_from_being_differentiated'),
   'object': rdflib.term.URIRef('http://jurisynth/data/as_between_the_suppliers_from_whom_the_purchaser_has_obtained_his_procurements_within_the_common_market_or_according_to_the_market_in_which_he_has_resold')}},
 {'doc_id': '31953D0030en.md',
  'chunk_id': 'chunk_2',
  'triple_id': 52,
  'triple': {'subject': rdflib.term.URIRef('http://jurisynth/data/conditions_of_sale'),
   'predicate': r

In [43]:
for name, value, triple in inspection_list[:100]:
    print(value)
    
    print(triple["subject"])
    print(triple["predicate"])
    print(repr(triple["object"]))
    
    print()


http://publications.europa.eu/ontology/cdm#restricted_by
http://jurisynth/data/seller
http://publications.europa.eu/ontology/cdm#restricted_by
rdflib.term.URIRef('http://jurisynth/data/more_favourable_periods_for_payment_than_those_calculable_from_the_price_list_and_conditions_of_sale_on_which_he_bases_his_quotation_unless_offset_by_a_corresponding_increase_in_price')

http://publications.europa.eu/ontology/cdm#procurement_public
http://jurisynth/data/the_foregoing_paragraph
http://jurisynth/data/shall_be_no_bar_to
rdflib.term.URIRef('http://publications.europa.eu/ontology/cdm#procurement_public')

http://publications.europa.eu/ontology/cdm#adopted_by
http://jurisynth/data/this_decision
http://publications.europa.eu/ontology/cdm#adopted_by
rdflib.term.URIRef('http://jurisynth/data/the_high_authority_at_its_meeting_on_2_may_1953')

http://publications.europa.eu/ontology/cdm#implements
http://jurisynth/data/decision_no_24-54
http://publications.europa.eu/ontology/cdm#implements
rdflib.te

### **Entity-Relation Resolver**

##### A. Helper Functions & Variables

In [ ]:
IDENTIFIER_PATTERNS = [
    # Legal structural references
    r"\barticle\s+\d+",
    r"\bart\.\s*\d+",
    r"\bparagraph\s+\d+",
    r"\bpara\.\s*\d+",
    r"\bpoint\s+\(?[a-z0-9]+\)?",
    r"\bsection\s+\d+",
    r"\bchapter\s+[ivxlcdm\d]+",
    r"\btitle\s+[ivxlcdm\d]+",
    r"\bannex\s+[ivxlcdm\d]+",
    r"\brecital\s+\d+",

    # Legal instrument identifiers
    r"\bno\.?\s*\d+",
    r"\b\d+/\d+\b",

    # Directive / regulation style identifiers
    r"\b\d{4}/\d+\b",

    # Standalone year identifiers
    r"\b(19|20)\d{2}\b",

    # Explicit subdivisions
    r"\([a-z]\)",
    r"\(\d+\)",
]

IDENTIFIER_REGEX = [
    re.compile(
        pattern,
        flags=re.IGNORECASE
    )
    for pattern in IDENTIFIER_PATTERNS
]

resolution_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "cluster_id": {
                "type": "integer"
            },
            "resolutions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "canonical_label": {
                            "type": "string"
                        },
                        "members": {
                            "type": "array",
                            "items": {
                                "type": "string"
                            },
                            "minItems": 1
                        }
                    },
                    "required": [
                        "canonical_label",
                        "members"
                    ],
                    "additionalProperties": False
                }
            }
        },
        "required": [
            "cluster_id",
            "resolutions"
        ],
        "additionalProperties": False
    }
}

resolution_prompt = """
You are resolving potentially duplicate labels extracted from EU legal documents.

Your task is to decide whether labels within each cluster refer to the same resource.

Rules:

1. This is a clustering task, not a rewriting task.
- The canonical_label MUST be copied exactly from one of the provided members.
- Do not create, modify, or improve labels.

2. Each cluster must be partitioned correctly.
- Every input label must appear exactly once in the output.
- Do not assign a label to multiple groups.
- Do not omit any labels.

3. Merge labels only when they clearly refer to the same resource.
- If uncertain, keep labels separate.
- A cluster being provided does not mean all labels should be merged.


Example:

Cluster ID: 5
Resource type: entity

Candidates:

Resource ID: c5_r1
Label: the member

Resource ID: c5_r2
Label: a member

Output:
{
    "cluster_id": 5,
    "resolutions": [
        {
            "canonical_label": "c5_r1",
            "members": ["c5_r1", "c5_r2"]
        }
    ]
                    
    
}

Now resolve the following clusters:

"""


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(
                self.parent[x]
            )
        return self.parent[x]

    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)

        if ra != rb:
            self.parent[rb] = ra

def normalize_key(label: str) -> str:
    """
    Normalize labels for clustering/matching.

    This function should only remove superficial differences:
    - casing
    - Unicode variants
    - whitespace inconsistencies
    - punctuation formatting
    - legal identifier spacing

    It should NOT attempt semantic equivalence.
    """

    # Case normalization
    label = label.casefold()

    # Normalize Unicode representations
    label = unicodedata.normalize("NFKC", label)

    # Replace common invisible whitespace characters
    label = label.replace("\u00a0", " ")   # non-breaking space
    label = label.replace("\u202f", " ")   # narrow no-break space
    label = label.replace("\u200b", "")    # zero-width space

    # Normalize dash variants
    label = re.sub(
        r"[\u2010\u2011\u2012\u2013\u2014\u2212]",
        "-",
        label
    )

    # Collapse whitespace
    label = re.sub(r"\s+", " ", label)

    # Normalize numeric subdivisions:
    # Article 93 (3) -> Article 93(3)
    # Point 5 (1) -> Point 5(1)
    label = re.sub(
        r"(\d+)\s+\(\s*(\d+)\s*\)",
        r"\1(\2)",
        label
    )

    # Normalize alphabetic subdivisions:
    # Article 5 (a) -> Article 5(a)
    label = re.sub(
        r"(\d+)\s+\(\s*([a-z])\s*\)",
        r"\1(\2)",
        label
    )

    # Clean spaces inside parentheses:
    # ( EEC ) -> (eec)
    # ( 3 ) -> (3)
    label = re.sub(r"\(\s+", "(", label)
    label = re.sub(r"\s+\)", ")", label)

    return label.strip()

def has_identifier_1(text: str) -> bool:
    """
    Returns True if the label looks like a legal identifier
    or numbered designation.

    Conservative by design.
    """

    return bool(
        re.search(
            r"""
            \d                      # any digit
            |
            \([A-Za-z0-9]+\)        # (1), (a), (ii)
            |
            \b[IVXLCDM]+\b          # Roman numerals
            """,
            text,
            flags=re.IGNORECASE | re.VERBOSE,
        )
    )

def extract_uri_label(uri: URIRef):
    """
    Extract readable local name from a URI.

    Example:
        http://jurisynth/data/decision_no_30
            ->
        decision no 30
    """

    try:
        _, local = split_uri(uri)
    except Exception:
        local = str(uri).rsplit("/", 1)[-1]

    return local.replace("_", " ")

def prepare_document_resources(scored_assertions):
    """
    Prepare custom assertion resources for per-document deduplication.

    Only the subject, predicate, and object of each assertion are
    considered for resource resolution. Modifiers are preserved
    separately and are not processed here.

    Returns
    -------
    tuple[defaultdict, defaultdict]

        document_entities:
        {
            doc_id:
            {
                uri:
                {
                    "uri": str,
                    "label": str,
                    "occurrences": [...],
                    "contexts": set(),
                    "neighbors": set()
                }
            }
        }

        document_relations:
        Same structure, but for predicate resources.
    """

    document_entities = defaultdict(dict)
    document_relations = defaultdict(dict)

    js_namespace = str(JS_DATA)

    for element in scored_assertions:

        doc_id = element["doc_id"]
        chunk_id = element["chunk_id"]
        assertion_id = element["assertion_id"]
        assertion = element["assertion"]

        for component in (
            "subject",
            "predicate",
            "object"
        ):

            value = assertion[component]

            # Only URI resources can require resolution.
            if not isinstance(value, URIRef):
                continue

            # Legal identifiers are deliberately excluded from
            # automatic deduplication.
            if (
                component != "predicate"
                and has_identifier_1(str(value))
            ):
                continue

            # Separate entity and relation resources.
            target = (
                document_relations
                if component == "predicate"
                else document_entities
            )

            uri = str(value)

            # Only custom Jurisynth resources are candidates
            # for document-level deduplication.
            if not uri.startswith(js_namespace):
                continue

            if uri not in target[doc_id]:

                target[doc_id][uri] = {
                    "uri": uri,
                    "label": extract_uri_label(value),
                    "occurrences": list(),
                    "contexts": set(),
                    "neighbors": set()
                }

            target[doc_id][uri]["occurrences"].append(
                {
                    "assertion_id": assertion_id,
                    "chunk_id": chunk_id,
                    "component": component
                }
            )

    return (
        document_entities,
        document_relations
    )

def ngram_tokenize(text, n=3):
    text = text.lower().replace(" ", "")

    return [
        text[i:i+n]
        for i in range(len(text)-n+1)
    ]

def attach_resource_embeddings(
    document_resources,
    emb_model,
    batch_size=128
):
    """
    Compute embeddings for all resources across
    all documents.

    Adds:
        resource["embedding"]

    Parameters
    ----------
    document_resources : dict
        {
            doc_id:
                {
                    uri:
                        resource_dict
                }
        }

    """

    resource_entries = list()
    labels = list()


    for doc_id, resources in document_resources.items():
        for uri, resource in resources.items():
            resource_entries.append(
                (
                    doc_id,
                    uri,
                    resource
                )
            )

            labels.append(resource["label"])

    embeddings = emb_model.encode(
        labels,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    for (_, _, resource), embedding in zip(
        resource_entries,
        embeddings
    ):

        resource["embedding"] = embedding

def build_candidate_clusters(
    doc_resources,
    similarity_threshold=0.9,
):
    """
    Build candidate equivalence clusters using
    pairwise cosine similarity.

    Parameters
    ----------
    doc_resources : dict

        Mapping:

        URIRef ->
        {
            "uri": URIRef,
            "label": str,
            "embedding": np.ndarray,
            "occurrences": list,
            "contexts": set,
            "neighbors": set
        }


    similarity_threshold : float
        Minimum cosine similarity required
        to merge two resources.


    Returns
    -------
    list[dict]

        [
            {
                "cluster_id": int,
                "resources": {
                    URIRef:
                        resource_dict
                }
            }
        ]
    """


    if not doc_resources:
        return list()


    uris = list(
        doc_resources.keys()
    )


    # ---------------------------------
    # Collect embeddings
    # ---------------------------------

    embeddings = np.stack(
        [
            doc_resources[uri]["embedding"]
            for uri in uris
        ]
    )


    # ---------------------------------
    # Pairwise cosine similarity
    #
    # embeddings are already normalized
    # ---------------------------------

    similarity_matrix = (
        embeddings @ embeddings.T
    )


    # ---------------------------------
    # Union-Find clustering
    # ---------------------------------

    uf = UnionFind(
        len(uris)
    )


    for i in range(len(uris)):

        for j in range(
            i + 1,
            len(uris)
        ):

            if (
                similarity_matrix[i, j]
                >= similarity_threshold
            ):

                uf.union(
                    i,
                    j
                )


    # ---------------------------------
    # Connected components
    # ---------------------------------

    grouped = defaultdict(list)


    for idx in range(len(uris)):

        grouped[
            uf.find(idx)
        ].append(idx)



    clusters = list()

    cluster_id = 1


    for members in grouped.values():

        # Ignore singleton resources
        if len(members) < 2:
            continue


        cluster_resources = dict()


        for idx in members:

            uri = uris[idx]

            cluster_resources[uri] = (
                doc_resources[uri]
            )


        clusters.append(
            {
                "cluster_id": cluster_id,
                "resources": cluster_resources
            }
        )

        cluster_id += 1


    return clusters

def build_resolution_batches(
    queries,
    batch_size=10
):
    """
    Convert resolution queries into batched LLM requests.

    Each batch contains:
        - query: string sent to the LLM
        - lookup: mapping from
          (doc_id, cluster_type, cluster_id)
          to temporary resource ID -> URI

    Parameters
    ----------
    queries : list[dict]

        Expected format:

        {
            "doc_id": str,
            "cluster_type": str,
            "cluster_id": int,
            "resources": {
                uri: {
                    "label": str,
                    ...
                }
            },
            "query": str
        }

    batch_size : int
        Number of clusters per LLM request.

    Returns
    -------
    list[dict]
    """

    batches = list()

    for batch_start in range(
        0,
        len(queries),
        batch_size
    ):

        batch = queries[
            batch_start:
            batch_start + batch_size
        ]

        query_parts = list()
        lookup = dict()

        for cluster_query in batch:
            doc_id = cluster_query["doc_id"]
            cluster_type = cluster_query["cluster_type"]
            cluster_id = cluster_query["cluster_id"]

            key = (
                doc_id,
                cluster_type,
                cluster_id
            )

            resource_map = dict()

            for idx, (uri, resource) in enumerate(
                cluster_query["resources"].items(),
                start=1
            ):
                temp_id = f"r{idx}"
                resource_map[temp_id] = uri

            lookup[key] = resource_map
            query_parts.append(cluster_query["query"])

        batches.append(
            {
                "query": "\n\n".join(query_parts),
                "lookup": lookup
            }
        )

    return batches

def has_identifier_2(label: str) -> bool:
    """
    Returns True if a resource label contains an
    identifier-like component.

    Identifier-bearing entities are treated as
    unsafe for automatic deduplication because
    small differences may represent genuinely
    different legal concepts.

    Examples:

        Article 5
        Article 6
        Decision No 30/53
        Directive 2004/18/EC
        Annex III

    """

    if not label:
        return False


    label = label.strip()


    return any(
        regex.search(label)
        for regex in IDENTIFIER_REGEX
    )

def filter_resolution_clusters(clusters):
    """
    Split candidate equivalence clusters into:

    review_clusters:
        Clusters that can proceed to semantic
        resolution.

    skipped_clusters:
        Clusters where every resource contains an
        identifier-like component and should not be
        merged automatically.


    Parameters
    ----------
    clusters : list[dict]

        Expected format:

        {
            "cluster_id": int,
            "resources": {
                URIRef:
                    {
                        "uri": URIRef,
                        "label": str,
                        ...
                    }
            }
        }


    Returns
    -------
    tuple[list, list]

        review_clusters,
        skipped_clusters

    """


    review_clusters = list()
    skipped_clusters = list()


    for cluster in clusters:

        labels = [
            resource["label"]
            for resource in cluster["resources"].values()
        ]

        # Conservative rule:
        # only skip when ALL resources are
        # identifier-bearing.
        if labels and all(
            has_identifier_2(label)
            for label in labels
        ):

            skipped_clusters.append(cluster)

        else:

            review_clusters.append(cluster)


    return (
        review_clusters,
        skipped_clusters
    )

def build_resolution_query(cluster, cluster_type, doc_id):
    """
    Build an LLM resolution query for a candidate
    equivalence cluster.

    Parameters
    ----------
    cluster : dict
        Candidate cluster containing resources.

    cluster_type : str
        "entity" or "relation"
    
    doc_id : str
        Source document identifier.

    Returns
    -------
    str
        Structured query string for LLM resolution.
    """

    lines = [
        f"Document: {doc_id}",
        f"Resource type: {cluster_type}",
        f"Cluster ID: {cluster['cluster_id']}",
        "",
        "Candidate resources:"
    ]


    for idx, (uri, resource) in enumerate(
        cluster["resources"].items(),
        start=1
    ):

        lines.extend([
            f"{idx}.",
            f"URI: {uri}",
            f"Label: {resource['label']}",
            ""
        ])


    return "\n".join(lines)

async def resolution_worker(
    batch,
    semaphore,
    rate_lock,
    last_request_time,
    cooldown_until,
    current_rps,
    max_rps,
    max_backoff=30,
    min_rps=0.25,
    recovery_step=0.10,
):
    """
    Resolve one batch of candidate clusters.
    """

    attempt = 0

    while True:
        async with semaphore:
            await wait_for_rate_limit(
                rate_lock,
                last_request_time,
                cooldown_until,
                current_rps,
            )

            try:
                response = await get_completion(
                    system_prompt=resolution_prompt,
                    query=batch["query"],
                    schema=resolution_schema,
                )

                clusters = json.loads(response)

                # ------------------------------------------
                # Slowly recover request rate
                # ------------------------------------------

                async with rate_lock:
                    current_rps[0] = min(max_rps, current_rps[0] + recovery_step)

                return {
                    "success": True,
                    "clusters": clusters,
                    "lookup": batch["lookup"],
                }


            except Exception as e:

                error_text = str(e)

                # ------------------------------------------
                # Retry transient API failures
                # ------------------------------------------

                if (
                    "429" in error_text
                    or
                    "503" in error_text
                ):
                    backoff = min(2 ** attempt, max_backoff)
                    backoff += random.uniform(0, 1)

                    async with rate_lock:
                        cooldown_until[0] = max(cooldown_until[0], time.monotonic() + backoff)
                        current_rps[0] = max(min_rps, current_rps[0] / 2)
                        current_rate = current_rps[0]

                    print(
                        f"[Resolution] "
                        f"{error_text}\n"
                        f"Cooldown: {backoff:.1f}s | "
                        f"RPS: {current_rate:.2f}"
                    )

                    attempt += 1

                    continue


                # ------------------------------------------
                # Non-retryable failure
                # ------------------------------------------

                print(f"Resolution failed: {e}")

                return {
                    "success": False,
                    "clusters": list(),
                    "lookup": batch["lookup"],
                }

def attach_lookup_metadata(clusters, lookup):
    """
    Restore local metadata omitted from the LLM prompt.

    Parameters
    ----------
    clusters : list[dict]

    lookup : dict

        Expected format:
        {
            cluster_id: {
                "doc_id": ...,
                "cluster_type": ...,
                "resource_map": ...
            }
        }

    Returns
    -------
    list[dict]
    """

    for cluster in clusters:
        cluster_id = cluster["cluster_id"]

        matches = [
            (key, resource_map)
            for key, resource_map in lookup.items()
            if key[2] == cluster_id
            ]

        if not matches:
            raise KeyError(f"No lookup found for cluster {cluster_id}")

        key, resource_map = matches[0]

        doc_id, cluster_type, _ = key

        cluster["doc_id"] = doc_id
        cluster["cluster_type"] = cluster_type
        cluster["resource_map"] = resource_map

    return clusters

async def resolve_batches(
    batches,
    semaphore,
    requests_per_second=2,
    max_backoff=30,
):

    rate_lock = asyncio.Lock()

    last_request_time = [0.0]
    cooldown_until = [0.0]

    current_rps = [requests_per_second]


    tasks = [

        resolution_worker(
            batch,
            semaphore,
            rate_lock,
            last_request_time,
            cooldown_until,
            current_rps,
            requests_per_second,
            max_backoff,
        )

        for batch in batches
    ]

    batch_results = await asyncio.gather(*tasks)
    resolved = list()

    for result in batch_results:
        if not result["success"]:
            continue

        resolved.extend(
            attach_lookup_metadata(
                result["clusters"],
                result["lookup"],
            )
        )


    return resolved

def build_resolution_map(resolved_clusters):
    """
    Convert LLM cluster resolutions into:

        old_uri -> canonical_uri

    Parameters
    ----------
    resolved_clusters : list[dict]
        Enriched LLM outputs containing:

        {
            "cluster_id": int,
            "resolutions": [...],
            "resource_map": {
                temporary_id: URI
            }
        }

    Returns
    -------
    dict
        URI replacement map.
    """

    resolution_map = dict()

    for cluster in resolved_clusters:
        for resolution in cluster["resolutions"]:
            canonical_uri = resolution["canonical_label"]

            for member_uri in resolution["members"]:
                resolution_map[member_uri] = canonical_uri

    return resolution_map

def apply_resolution(
    scored_assertions,
    entity_map,
    relation_map
):
    """
    Apply URI resolution maps to scored assertions.

    Only assertion subject, predicate, and object components
    are resolved. Modifiers are carried through unchanged.

    Parameters
    ----------
    scored_assertions : list[dict]
        Assertions containing:
            - doc_id
            - chunk_id
            - assertion_id
            - assertion
            - modifiers

    entity_map : dict
        Mapping from old entity URI -> canonical entity URI.

    relation_map : dict
        Mapping from old relation URI -> canonical relation URI.

    Returns
    -------
    list[dict]
        Assertions with resolved assertion components and
        unchanged modifiers.
    """

    resolved_assertions = list()

    for element in scored_assertions:

        updated = element.copy()

        assertion = element["assertion"].copy()

        # --------------------------------------------------
        # Subject
        # --------------------------------------------------

        subject = assertion["subject"]

        if (
            isinstance(subject, URIRef)
            and subject in entity_map
        ):
            assertion["subject"] = entity_map[subject]

        # --------------------------------------------------
        # Object
        # --------------------------------------------------

        object_value = assertion["object"]

        if (
            isinstance(object_value, URIRef)
            and object_value in entity_map
        ):
            assertion["object"] = entity_map[object_value]

        # --------------------------------------------------
        # Predicate
        # --------------------------------------------------

        predicate = assertion["predicate"]

        if (
            isinstance(predicate, URIRef)
            and predicate in relation_map
        ):
            assertion["predicate"] = relation_map[predicate]

        # --------------------------------------------------
        # Preserve assertion + modifiers
        # --------------------------------------------------

        updated["assertion"] = assertion

        # modifiers remains untouched because it is outside
        # the assertion object.

        resolved_assertions.append(updated)

    return resolved_assertions

##### B. Execution

In [ ]:
assertion_lookup = {
    element["assertion_id"]: element["assertion"]
    for element in scored_assertions
}

document_entities, document_relations = prepare_document_resources(scored_assertions)

attach_resource_embeddings(document_entities, emb_model)
attach_resource_embeddings(document_relations, emb_model)

entity_clusters = {
    doc_id: build_candidate_clusters(resources)
    for doc_id, resources in document_entities.items()
}

relation_clusters = {
    doc_id: build_candidate_clusters(resources)
    for doc_id, resources in document_relations.items()
}

INFO:sentence_transformers.base.model:No device provided, using cpu
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Re

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
resolution_queries = list()
skipped_resolution_clusters = list()

for cluster_type, cluster_collection in [
    ("entity", entity_clusters),
    ("relation", relation_clusters)
]:

    for doc_id, clusters in cluster_collection.items():

        # ------------------------------------------
        # Filter identifier-heavy clusters
        # ------------------------------------------

        review_clusters, skipped_clusters = (
            filter_resolution_clusters(clusters)
        )

        skipped_resolution_clusters.extend(skipped_clusters)

        # ------------------------------------------
        # Build LLM queries
        # ------------------------------------------

        for cluster in review_clusters:

            resolution_queries.append(
                {
                    "doc_id": doc_id,
                    "cluster_type": cluster_type,
                    "cluster_id": cluster["cluster_id"],

                    # Keep original resources
                    "resources": cluster["resources"],

                    # Human-readable prompt
                    "query": build_resolution_query(
                        cluster,
                        cluster_type,
                        doc_id
                    )
                }
            )

entity_queries = [
    q for q in resolution_queries
    if q["cluster_type"] == "entity"
]

relation_queries = [
    q for q in resolution_queries
    if q["cluster_type"] == "relation"
]

entity_query_batches = build_resolution_batches(entity_queries, batch_size=10)
relation_query_batches = build_resolution_batches(relation_queries, batch_size=10)

In [ ]:
resolved_entities = await resolve_batches(entity_query_batches, SEMAPHORE)
resolved_relations = await resolve_batches(relation_query_batches, SEMAPHORE)

In [ ]:
# ---------------------------------------------------------------------
# Generate URI resolution maps
# ---------------------------------------------------------------------

entity_map = build_resolution_map(resolved_entities)
relation_map = build_resolution_map(resolved_relations)

# ---------------------------------------------------------------------
# Apply resolutions
# ---------------------------------------------------------------------

resolved_triples = apply_resolution(
    scored_triples,
    entity_map,
    relation_map
)

### **Assertion Validator**

##### A. Helper Functions & Variables

In [ ]:
# ---------------------------------------------------------------------
# Built-in predicates
# ---------------------------------------------------------------------

BUILTIN_PREDICATES = {
    RDF.type,
    RDFS.label,
    RDFS.comment,
}


# ---------------------------------------------------------------------
# Resource type lookup
# ---------------------------------------------------------------------

def build_resource_type_lookup(scored_assertions):
    """
    Build a lookup table:

        resource URI -> set[rdf:type]

    A resource may have multiple rdf:type assertions.
    """

    resource_types = dict()

    for element in scored_assertions:
        assertion = element["assertion"]

        if assertion["predicate"] != RDF.type:
            continue

        resource = assertion["subject"]
        rdf_type = assertion["object"]

        resource_types.setdefault(
            resource,
            set()
        ).add(rdf_type)

    return resource_types


# ---------------------------------------------------------------------
# Main validation function
# ---------------------------------------------------------------------

def validate_assertions(
    resolved_assertions,
    resource_metadata
):
    """
    Validate resolved assertions against the ontology schema.

    Validation stages
    -----------------
        • Predicate existence
        • Subject domain
        • Object range
        • Literal datatype

    Missing rdf:type assertions are treated as warnings,
    not errors.

    Modifiers are preserved unchanged and are not validated here.
    """

    resource_types = build_resource_type_lookup(
        resolved_assertions
    )

    validated_assertions = list()
    validation_errors = list()

    statistics = Counter()

    for element in resolved_assertions:

        statistics["total"] += 1

        assertion = element["assertion"]

        subject = assertion["subject"]
        predicate = assertion["predicate"]
        obj = assertion["object"]

        errors = list()
        warnings = list()

        # ---------------------------------------------------------
        # Built-in RDF predicates
        # ---------------------------------------------------------

        if predicate in BUILTIN_PREDICATES:

            updated = {
                **element,
                "validation_status": "valid",
                "warnings": [],
            }

            validated_assertions.append(updated)
            statistics["valid"] += 1

            continue

        # ---------------------------------------------------------
        # Predicate existence
        # ---------------------------------------------------------

        metadata = resource_metadata.get(
            str(predicate)
        )

        if metadata is None:

            warnings.append(
                "unknown_predicate"
            )

        else:

            # =====================================================
            # Domain validation
            # =====================================================

            expected_domains = metadata.get(
                "domain",
                set()
            )

            if expected_domains:

                subject_types = resource_types.get(
                    subject
                )

                if not subject_types:

                    warnings.append(
                        "missing_subject_type"
                    )

                elif subject_types.isdisjoint(
                    expected_domains
                ):

                    errors.append(
                        "domain_violation"
                    )

            # =====================================================
            # Range validation
            # =====================================================

            expected_ranges = metadata.get(
                "range",
                set()
            )

            property_type = metadata.get(
                "type"
            )

            # -----------------------------------------------------
            # Object property
            # -----------------------------------------------------

            if property_type == "object property":

                if not isinstance(
                    obj,
                    URIRef
                ):

                    errors.append(
                        "expected_resource"
                    )

                elif expected_ranges:

                    object_types = resource_types.get(
                        obj
                    )

                    if not object_types:

                        warnings.append(
                            "missing_object_type"
                        )

                    elif object_types.isdisjoint(
                        expected_ranges
                    ):

                        errors.append(
                            "range_violation"
                        )

            # -----------------------------------------------------
            # Datatype property
            # -----------------------------------------------------

            elif property_type == "datatype property":

                if not isinstance(
                    obj,
                    Literal
                ):

                    errors.append(
                        "expected_literal"
                    )

                elif expected_ranges:

                    literal_datatype = obj.datatype

                    if (
                        literal_datatype
                        not in expected_ranges
                    ):

                        errors.append(
                            "datatype_violation"
                        )

        # ---------------------------------------------------------
        # Finalise
        # ---------------------------------------------------------

        if errors:

            statistics["invalid"] += 1

            for error in errors:
                statistics[error] += 1

            updated = {
                **element,
                "validation_status": "invalid",
                "warnings": warnings,
                "errors": errors,
            }

            validation_errors.append(
                updated
            )

        else:

            status = (
                "warning"
                if warnings
                else "valid"
            )

            statistics[status] += 1

            for warning in warnings:
                statistics[warning] += 1

            updated = {
                **element,
                "validation_status": status,
                "warnings": warnings,
            }

            validated_assertions.append(
                updated
            )

    return {
        "validated_assertions": validated_assertions,
        "validation_errors": validation_errors,
        "statistics": dict(statistics),
    }

##### B. Execution

In [ ]:
validation = validate_assertions(resolved_assertions, resource_metadata)

validated_assertions = validation["validated_assertions"]
validation_errors = validation["validation_errors"]
stats = validation["statistics"]

print(stats)

{'total': 4814, 'warning': 4066, 'unknown_predicate': 4063, 'valid': 747, 'missing_subject_type': 3, 'missing_object_type': 3, 'invalid': 1, 'domain_violation': 1}


### **Graph Serializer**

##### A. Helper Functions & Variables

In [ ]:
from collections import defaultdict
import re

from rdflib import (
    Dataset,
    Namespace,
    RDF,
    RDFS,
    URIRef,
    Literal,
)


# ---------------------------------------------------------------------
# Namespaces
# ---------------------------------------------------------------------

JS_DATA = Namespace("http://jurisynth/data/")
JS_SOURCE = Namespace("http://jurisynth/source/")

DOCUMENT = Namespace("http://jurisynth/source/document/")
CHUNK = Namespace("http://jurisynth/source/chunk/")
ASSERTION = Namespace("http://jurisynth/source/assertion/")


# ---------------------------------------------------------------------
# URI helpers
# ---------------------------------------------------------------------

def normalize_identifier(text):
    """
    Convert an arbitrary identifier into a URI-safe fragment.
    """

    text = str(text).strip().lower()

    # Remove file extension
    text = re.sub(r"\.[^.]+$", "", text)

    # Replace non-alphanumeric runs
    text = re.sub(r"[^a-z0-9]+", "_", text)

    # Collapse repeated underscores
    text = re.sub(r"_+", "_", text)

    return text.strip("_")


def document_uri(doc_id):
    return DOCUMENT[
        normalize_identifier(doc_id)
    ]


def chunk_uri(doc_id, chunk_id):
    return CHUNK[
        f"{normalize_identifier(doc_id)}_"
        f"{normalize_identifier(chunk_id)}"
    ]


def assertion_uri(doc_id, chunk_id, assertion_id):
    """
    Generate a stable URI for one extracted assertion.

    The chunk ID is included to make the identifier globally
    traceable to its source location.
    """

    return ASSERTION[
        f"{normalize_identifier(doc_id)}_"
        f"{normalize_identifier(chunk_id)}_"
        f"{normalize_identifier(assertion_id)}"
    ]


def modifier_uri(doc_id, chunk_id, assertion_id, modifier_id):
    """
    Generate a stable URI for one modifier belonging to
    an assertion.
    """

    return JS_DATA[
        "modifier_"
        f"{normalize_identifier(doc_id)}_"
        f"{normalize_identifier(chunk_id)}_"
        f"{normalize_identifier(assertion_id)}_"
        f"{normalize_identifier(modifier_id)}"
    ]


# ---------------------------------------------------------------------
# Modifier serialization
# ---------------------------------------------------------------------

def serialize_modifier(
    graph,
    modifier_resource,
    modifier,
):
    """
    Serialize one modifier into the assertion graph.

    At present, modifiers are treated conservatively as opaque
    values because the extraction pipeline does not yet impose a
    structured modifier schema.

    A future structured modifier representation can be introduced
    here without changing the assertion graph architecture.
    """

    graph.add(
        (
            modifier_resource,
            RDF.type,
            JS_SOURCE.Modifier,
        )
    )

    graph.add(
        (
            modifier_resource,
            JS_SOURCE.value,
            Literal(str(modifier)),
        )
    )


# ---------------------------------------------------------------------
# Dataset builder
# ---------------------------------------------------------------------

def build_quad_dataset(
    validated_assertions,
    schema_namespaces=None,
):
    """
    Construct the Jurisynth RDF Dataset from validated assertions.

    Architecture
    ------------

    1. Chunk graphs contain the ordinary RDF triples extracted
       from each chunk.

    2. A single global assertion graph contains assertion-level
       n-ary structures when an assertion has modifiers.

    3. Document graphs describe document provenance and link
       documents to their chunks and assertion resources.

    Conceptually:

        GRAPH chunk_X {
            subject predicate object .
        }

        GRAPH js:Assertions {
            assertion_X rdf:type js:Assertion ;
                js:subject subject ;
                js:predicate predicate ;
                js:object object ;
                js:sourceChunk chunk_X ;
                js:hasModifier modifier_X .

            modifier_X rdf:type js:Modifier ;
                js:value "..." .
        }

        GRAPH document_X {
            document_X rdf:type js:Document ;
                js:hasChunk chunk_X ;
                js:hasAssertion assertion_X .
        }

    The ordinary triple is therefore retained for efficient graph
    traversal and retrieval, while the assertion representation
    provides a place for statement-level metadata.

    Parameters
    ----------
    validated_assertions : list[dict]

        Expected format:

        {
            "doc_id": str,
            "chunk_id": str,
            "assertion_id": int,

            "assertion": {
                "subject": URIRef,
                "predicate": URIRef,
                "object": URIRef | Literal
            },

            "modifiers": [...]
        }

    schema_namespaces : list[(prefix, Namespace)], optional
        Ontology/schema namespace bindings.

    Returns
    -------
    rdflib.Dataset
    """

    dataset = Dataset()

    # -------------------------------------------------------------
    # Namespace bindings
    # -------------------------------------------------------------

    dataset.bind("rdf", RDF)
    dataset.bind("rdfs", RDFS)

    dataset.bind("source", JS_SOURCE)
    dataset.bind("data", JS_DATA)

    dataset.bind("document", DOCUMENT)
    dataset.bind("chunk", CHUNK)
    dataset.bind("assertion", ASSERTION)

    if schema_namespaces:

        for prefix, namespace in schema_namespaces:

            dataset.bind(
                prefix,
                namespace,
            )

    # -------------------------------------------------------------
    # Global assertion graph
    # -------------------------------------------------------------

    assertion_graph = dataset.graph(ASSERTION)

    # -------------------------------------------------------------
    # Track discovered chunks and assertions
    # -------------------------------------------------------------

    document_chunks = defaultdict(set)
    document_assertions = defaultdict(set)

    # -------------------------------------------------------------
    # Serialize validated assertions
    # -------------------------------------------------------------

    for element in validated_assertions:

        doc_id = element["doc_id"]
        chunk_id = element["chunk_id"]
        assertion_id = element["assertion_id"]

        assertion = element["assertion"]

        # IMPORTANT:
        # modifiers lives alongside "assertion", not inside it.
        modifiers = element.get(
            "modifiers",
            [],
        )

        # ---------------------------------------------------------
        # Source chunk
        # ---------------------------------------------------------

        source_chunk = chunk_uri(
            doc_id,
            chunk_id,
        )

        chunk_graph = dataset.graph(
            source_chunk
        )

        # ---------------------------------------------------------
        # Core RDF triple
        #
        # Always preserve this representation.
        # ---------------------------------------------------------

        chunk_graph.add(
            (
                assertion["subject"],
                assertion["predicate"],
                assertion["object"],
            )
        )

        document_chunks[doc_id].add(
            chunk_id
        )

        # ---------------------------------------------------------
        # Assertion-level representation
        #
        # Only required when assertion-specific metadata exists.
        # Currently that means modifiers.
        # ---------------------------------------------------------

        if not modifiers:
            continue

        assertion_resource = assertion_uri(
            doc_id,
            chunk_id,
            assertion_id,
        )

        # ---------------------------------------------------------
        # Assertion identity
        # ---------------------------------------------------------

        assertion_graph.add(
            (
                assertion_resource,
                RDF.type,
                JS_SOURCE.Assertion,
            )
        )

        # ---------------------------------------------------------
        # Core assertion components
        # ---------------------------------------------------------

        assertion_graph.add(
            (
                assertion_resource,
                JS_SOURCE.subject,
                assertion["subject"],
            )
        )

        assertion_graph.add(
            (
                assertion_resource,
                JS_SOURCE.predicate,
                assertion["predicate"],
            )
        )

        assertion_graph.add(
            (
                assertion_resource,
                JS_SOURCE.object,
                assertion["object"],
            )
        )

        # ---------------------------------------------------------
        # Provenance
        # ---------------------------------------------------------

        assertion_graph.add(
            (
                assertion_resource,
                JS_SOURCE.source_chunk,
                source_chunk,
            )
        )

        # ---------------------------------------------------------
        # Modifiers
        # ---------------------------------------------------------

        for modifier_id, modifier in enumerate(
            modifiers,
            start=1,
        ):

            modifier_resource = modifier_uri(
                doc_id,
                chunk_id,
                assertion_id,
                modifier_id,
            )

            assertion_graph.add(
                (
                    assertion_resource,
                    JS_SOURCE.has_modifier,
                    modifier_resource,
                )
            )

            serialize_modifier(
                assertion_graph,
                modifier_resource,
                modifier,
            )

        # Track assertion for document-level provenance.
        document_assertions[doc_id].add(
            assertion_resource
        )

    # -------------------------------------------------------------
    # Populate document graphs
    # -------------------------------------------------------------

    for doc_id, chunks in document_chunks.items():

        doc_resource = document_uri(
            doc_id
        )

        doc_graph = dataset.graph(
            doc_resource
        )

        # ---------------------------------------------------------
        # Document identity
        # ---------------------------------------------------------

        doc_graph.add(
            (
                doc_resource,
                RDF.type,
                JS_SOURCE.Document,
            )
        )

        doc_graph.add(
            (
                doc_resource,
                RDFS.label,
                Literal(str(doc_id)),
            )
        )

        # ---------------------------------------------------------
        # Chunk resources
        # ---------------------------------------------------------

        for chunk_id in sorted(chunks):

            chunk_resource = chunk_uri(
                doc_id,
                chunk_id,
            )

            doc_graph.add(
                (
                    chunk_resource,
                    RDF.type,
                    JS_SOURCE.Chunk,
                )
            )

            doc_graph.add(
                (
                    doc_resource,
                    JS_SOURCE.has_chunk,
                    chunk_resource,
                )
            )

    # -------------------------------------------------------------
    # Link assertions from document provenance graphs
    # -------------------------------------------------------------

    for doc_id, assertions in document_assertions.items():

        doc_resource = document_uri(
            doc_id
        )

        doc_graph = dataset.graph(
            doc_resource
        )

        for assertion_resource in assertions:

            doc_graph.add(
                (
                    doc_resource,
                    JS_SOURCE.has_assertion,
                    assertion_resource,
                )
            )

    return dataset


# ---------------------------------------------------------------------
# Serialisation
# ---------------------------------------------------------------------

def serialize_dataset(
    dataset,
    output_file,
    format="nquads",
):
    """
    Serialize an RDF Dataset.

    Parameters
    ----------
    dataset : rdflib.Dataset

    output_file : str

    format : str
        Defaults to N-Quads.
    """

    dataset.serialize(
        destination=output_file,
        format=format,
    )

##### B. Execution

In [ ]:
dataset = build_quad_dataset(
    validated_assertions,
    schema_namespaces=schema_namespaces
)

serialize_dataset(dataset, "jurisynth_graph.nq")

http://jurisynth/source/chunk/31960r0011en_chunk_3
http://jurisynth/source/document/31963d0491en
http://jurisynth/source/chunk/31961d1009en_chunk_8
http://jurisynth/source/chunk/31961d1009en_chunk_7
http://jurisynth/source/document/31961r0007en
http://jurisynth/source/chunk/31953d0030en_chunk_2
http://jurisynth/source/chunk/31961r0007en_chunk_1
http://jurisynth/source/chunk/31961x1201en_chunk_2
http://jurisynth/source/chunk/31960r0011en_chunk_5
http://jurisynth/source/chunk/31962l0302en_chunk_3
http://jurisynth/source/chunk/31963d0491en_chunk_1
http://jurisynth/source/chunk/31963d0019_01_en_chunk_2
http://jurisynth/source/document/31954s0024en
http://jurisynth/source/chunk/31958r0003_01_en_chunk_2
http://jurisynth/source/chunk/31958q1006en_chunk_3
http://jurisynth/source/chunk/31960r0011en_chunk_4
http://jurisynth/source/document/31963l0474en
http://jurisynth/source/chunk/31958q1006en_chunk_2
http://jurisynth/source/chunk/31961d1009en_chunk_2
http://jurisynth/source/chunk/31963d0019_01

##### C. Inspection

In [ ]:
gen_graph = rdflib.Dataset().parse("./jurisynth_graph.nq", format="nquads")

sum = 0

for graph in gen_graph.graphs():
    print(
        graph.identifier,
        len(graph)
    )
    sum += len(graph)

print(sum)

http://jurisynth/source/document/31963d0491en 4
http://jurisynth/source/chunk/31960r0011en_chunk_3 54
http://jurisynth/source/chunk/31961d1009en_chunk_8 140
http://jurisynth/source/chunk/31961d1009en_chunk_7 62
http://jurisynth/source/document/31961r0007en 4
http://jurisynth/source/chunk/31961r0007en_chunk_1 20
http://jurisynth/source/chunk/31953d0030en_chunk_2 58
http://jurisynth/source/chunk/31961x1201en_chunk_2 48
http://jurisynth/source/chunk/31960r0011en_chunk_5 23
http://jurisynth/source/chunk/31962l0302en_chunk_3 191
http://jurisynth/source/chunk/31963d0491en_chunk_1 22
http://jurisynth/source/chunk/31963d0019_01_en_chunk_2 46
http://jurisynth/source/document/31954s0024en 4
http://jurisynth/source/chunk/31958r0003_01_en_chunk_2 57
http://jurisynth/source/chunk/31958q1006en_chunk_3 26
http://jurisynth/source/chunk/31960r0011en_chunk_4 75
http://jurisynth/source/document/31963l0474en 4
http://jurisynth/source/chunk/31958q1006en_chunk_2 94
http://jurisynth/source/chunk/31961d1009en

In [79]:
query = """
SELECT ?graph ?doc ?chunk
WHERE {

    GRAPH ?graph {

        ?doc <http://jurisynth/source/has_chunk> ?chunk .

    }

}
"""


for row in gen_graph.query(query):

    print(
        "Graph:",
        row.graph
    )

    print(
        "Document:",
        row.doc
    )

    print(
        "Chunk:",
        row.chunk
    )

    print()

Graph: http://jurisynth/source/document/31963d0491en
Document: http://jurisynth/source/document/31963d0491en
Chunk: http://jurisynth/source/chunk/31963d0491en_chunk_1

Graph: http://jurisynth/source/document/31961r0007en
Document: http://jurisynth/source/document/31961r0007en
Chunk: http://jurisynth/source/chunk/31961r0007en_chunk_1

Graph: http://jurisynth/source/document/31954s0024en
Document: http://jurisynth/source/document/31954s0024en
Chunk: http://jurisynth/source/chunk/31954s0024en_chunk_1

Graph: http://jurisynth/source/document/31963l0474en
Document: http://jurisynth/source/document/31963l0474en
Chunk: http://jurisynth/source/chunk/31963l0474en_chunk_1

Graph: http://jurisynth/source/document/31961d0408_01_en
Document: http://jurisynth/source/document/31961d0408_01_en
Chunk: http://jurisynth/source/chunk/31961d0408_01_en_chunk_2

Graph: http://jurisynth/source/document/31954s0026en
Document: http://jurisynth/source/document/31954s0026en
Chunk: http://jurisynth/source/chunk/31

### **Community Constructor**

##### A Helper Functions & Variables

In [ ]:
from collections import defaultdict

import igraph as ig
import leidenalg as la

from rdflib import (
    Dataset,
    Namespace,
    RDF,
    RDFS,
    URIRef,
)


# ---------------------------------------------------------------------
# Namespaces
# ---------------------------------------------------------------------

JS_SOURCE = Namespace("http://jurisynth/source/")
JS_DATA = Namespace("http://jurisynth/data/")

DOCUMENT = Namespace("http://jurisynth/source/document/")
CHUNK = Namespace("http://jurisynth/source/chunk/")


# ---------------------------------------------------------------------
# Default filtering
# ---------------------------------------------------------------------

DEFAULT_EXCLUDED_PREDICATES = {
    RDF.type,
    RDFS.label,
}


# ---------------------------------------------------------------------
# Dataset graph helpers
# ---------------------------------------------------------------------

def is_chunk_graph(graph_identifier):
    """
    Return True if the graph identifier belongs to a source chunk graph.

    Only chunk graphs contain the semantic RDF assertions used to
    construct the entity graph.

    Assertion/provenance graphs are deliberately excluded because their
    triples describe the representation of an assertion rather than the
    semantic relationship itself.
    """

    return str(graph_identifier).startswith(
        str(CHUNK)
    )


def extract_semantic_triples(
    dataset,
):
    """
    Extract semantic RDF triples from the completed KG.

    Only triples stored in chunk named graphs are considered.

    Returns
    -------
    list[tuple]
        A list of:
            (subject, predicate, object)

    Notes
    -----
    This intentionally ignores assertion/provenance graphs. A triple such
    as:

        assertion_007 -> subject -> :EntityA

    is provenance structure, not an EntityA relationship.

    The actual semantic assertion remains:

        :EntityA -> :someProperty -> :EntityB
    """

    semantic_triples = list()

    for graph in dataset.contexts():

        if not is_chunk_graph(
            graph.identifier
        ):
            continue

        for subject, predicate, obj in graph:

            semantic_triples.append(
                (
                    subject,
                    predicate,
                    obj,
                )
            )

    return semantic_triples


# ---------------------------------------------------------------------
# Entity / relation extraction
# ---------------------------------------------------------------------

def extract_entities_and_relations(
    dataset,
    excluded_predicates=None,
):
    """
    Derive the entity and relation sets from the completed KG.

    Entities:
        URI resources participating in semantic object-property
        relationships.

    Relations:
        Predicates connecting URI resources.

    Literal-valued assertions are not included in the entity graph,
    because literals cannot act as graph vertices.

    Returns
    -------
    tuple[set, set]
        entities, relations
    """

    if excluded_predicates is None:
        excluded_predicates = (
            DEFAULT_EXCLUDED_PREDICATES
        )

    entities = set()
    relations = set()

    semantic_triples = extract_semantic_triples(
        dataset
    )

    for subject, predicate, obj in semantic_triples:

        if predicate in excluded_predicates:
            continue

        if not isinstance(subject, URIRef):
            continue

        if not isinstance(obj, URIRef):
            continue

        entities.add(subject)
        entities.add(obj)
        relations.add(predicate)

    return entities, relations


# ---------------------------------------------------------------------
# Build entity graph
# ---------------------------------------------------------------------

def build_entity_graph(
    dataset,
    excluded_predicates=None,
):
    """
    Convert the completed KG into an unweighted entity graph.

    Nodes:
        RDF resources.

    Edges:
        Object-property relationships between RDF resources.

    Edge metadata:
        The RDF predicate responsible for the edge.

    Important:
        Edge multiplicity is deliberately NOT represented as a weight.

        If the same semantic triple occurs repeatedly across chunks,
        it still represents one graph relationship.
    """

    if excluded_predicates is None:
        excluded_predicates = (
            DEFAULT_EXCLUDED_PREDICATES
        )

    vertices = dict()
    edges = list()
    predicates = list()

    # Prevent duplicate graph edges while retaining predicate identity.
    seen_edges = set()

    semantic_triples = extract_semantic_triples(
        dataset
    )

    for subject, predicate, obj in semantic_triples:

        # -------------------------------------------------------------
        # Ignore schema/noise predicates
        # -------------------------------------------------------------

        if predicate in excluded_predicates:
            continue

        # -------------------------------------------------------------
        # Only URI -> URI relationships become entity-graph edges
        # -------------------------------------------------------------

        if not isinstance(subject, URIRef):
            continue

        if not isinstance(obj, URIRef):
            continue

        # -------------------------------------------------------------
        # Register vertices
        # -------------------------------------------------------------

        if subject not in vertices:
            vertices[subject] = len(vertices)

        if obj not in vertices:
            vertices[obj] = len(vertices)

        # -------------------------------------------------------------
        # Avoid edge multiplicity
        # -------------------------------------------------------------

        edge_key = (
            subject,
            predicate,
            obj,
        )

        if edge_key in seen_edges:
            continue

        seen_edges.add(edge_key)

        edges.append(
            (
                vertices[subject],
                vertices[obj],
            )
        )

        predicates.append(predicate)

    # -------------------------------------------------------------
    # Construct graph
    # -------------------------------------------------------------

    graph = ig.Graph(
        n=len(vertices),
        edges=edges,
        directed=False,
    )

    graph.vs["uri"] = list(
        vertices.keys()
    )

    graph.es["predicate"] = predicates

    return graph


# ---------------------------------------------------------------------
# Single Leiden pass
# ---------------------------------------------------------------------

def leiden_partition(
    graph,
    resolution=1.0,
    seed=42,
):
    """
    Run one Leiden partitioning pass.
    """

    return la.find_partition(
        graph,
        la.RBConfigurationVertexPartition,
        resolution_parameter=resolution,
        seed=seed,
    )


# ---------------------------------------------------------------------
# Build collapsed community graph
# ---------------------------------------------------------------------

def build_community_graph(
    graph,
    partition,
):
    """
    Collapse a graph's communities into a new graph.

    Each vertex in the resulting graph represents one community from
    the previous level.

    No edge weights are used. Multiple inter-community relationships
    therefore collapse into a single edge.
    """

    vertex_to_comm = dict()

    for community_id, members in enumerate(
        partition
    ):
        for vertex in members:
            vertex_to_comm[vertex] = community_id

    edges = list()
    seen_edges = set()

    for edge in graph.es:

        source = vertex_to_comm[
            edge.source
        ]

        target = vertex_to_comm[
            edge.target
        ]

        # Internal relationship
        if source == target:
            continue

        edge_key = tuple(
            sorted(
                (
                    source,
                    target,
                )
            )
        )

        if edge_key in seen_edges:
            continue

        seen_edges.add(edge_key)
        edges.append(edge_key)

    community_graph = ig.Graph(
        n=len(partition),
        edges=edges,
        directed=False,
    )

    return community_graph


# ---------------------------------------------------------------------
# Hierarchical Leiden
# ---------------------------------------------------------------------

def hierarchical_leiden(
    graph,
    max_levels=5,
    resolution=1.0,
    seed=42,
):
    """
    Generate a genuine hierarchical Leiden structure.

    Level 0
        Community members are original entity URIs.

    Level > 0
        Community members are the community URIs from the immediately
        preceding level.

    Each higher-level community therefore has an explicit parent-child
    relationship with the communities below it.

    Returns
    -------
    dict

        {
            0: {
                0: {
                    "members": [entity_uri, ...],
                    "children": [],
                    "parent": None
                }
            },

            1: {
                0: {
                    "members": [community_uri, ...],
                    "children": [...],
                    "parent": None
                }
            }
        }
    """

    hierarchy = dict()

    current_graph = graph

    # -----------------------------------------------------------------
    # At level 0, graph vertices represent actual entities.
    # At subsequent levels, they represent communities.
    # -----------------------------------------------------------------

    current_vertex_ids = [
        graph.vs[index]["uri"]
        for index in range(
            graph.vcount()
        )
    ]

    previous_community_uris = list()

    for level in range(max_levels):

        # -------------------------------------------------------------
        # Run Leiden
        # -------------------------------------------------------------

        partition = leiden_partition(
            current_graph,
            resolution=resolution,
            seed=seed,
        )

        community_count = len(partition)

        communities = dict()

        current_community_uris = list()

        # -------------------------------------------------------------
        # Construct communities
        # -------------------------------------------------------------

        for community_id, members in enumerate(
            partition
        ):

            community_uri = JS_DATA[
                f"community_l{level}_{community_id}"
            ]

            current_community_uris.append(
                community_uri
            )

            member_uris = [
                current_vertex_ids[vertex]
                for vertex in members
            ]

            communities[community_id] = {
                "uri": community_uri,
                "members": member_uris,
                "children": list(),
                "parent": None,
            }

        # -------------------------------------------------------------
        # Establish parent-child relationships
        #
        # At level > 0:
        #
        # current community
        #       |
        #       +-- child community from previous level
        #
        # -------------------------------------------------------------

        if level > 0:

            previous_to_parent = dict()

            for community_id, members in enumerate(
                partition
            ):

                parent_uri = current_community_uris[
                    community_id
                ]

                for vertex in members:

                    child_uri = current_vertex_ids[
                        vertex
                    ]

                    previous_to_parent[
                        child_uri
                    ] = parent_uri

                    communities[
                        community_id
                    ]["children"].append(
                        child_uri
                    )

            # ---------------------------------------------------------
            # Store parent information on previous-level communities
            # ---------------------------------------------------------

            for previous_community in hierarchy[
                level - 1
            ].values():

                previous_uri = (
                    previous_community["uri"]
                )

                previous_community[
                    "parent"
                ] = previous_to_parent.get(
                    previous_uri
                )

        hierarchy[level] = communities

        # -------------------------------------------------------------
        # Stop conditions
        # -------------------------------------------------------------

        if community_count <= 1:
            break

        # If the partition did not reduce the number of vertices,
        # there is no meaningful higher-level hierarchy.
        if community_count >= current_graph.vcount():
            break

        # -------------------------------------------------------------
        # Collapse communities into next-level graph
        # -------------------------------------------------------------

        current_graph = build_community_graph(
            current_graph,
            partition,
        )

        # At the next level, vertices represent the current
        # level's communities.
        current_vertex_ids = (
            current_community_uris
        )

    return hierarchy


# ---------------------------------------------------------------------
# Full community-construction pipeline
# ---------------------------------------------------------------------

def build_graph_communities(
    dataset,
    max_levels=5,
    resolution=1.0,
    excluded_predicates=None,
):
    """
    Construct global communities from the completed KG.

    The module derives its entities and semantic relationships directly
    from the Dataset rather than depending on intermediate pipeline
    variables.
    """

    graph = build_entity_graph(
        dataset,
        excluded_predicates=excluded_predicates,
    )

    hierarchy = hierarchical_leiden(
        graph,
        max_levels=max_levels,
        resolution=resolution,
    )

    return hierarchy, graph


# ---------------------------------------------------------------------
# Community RDF serialization
# ---------------------------------------------------------------------

def serialize_communities(
    hierarchy,
    dataset=None,
):
    """
    Serialize the community hierarchy into one dedicated named graph.

    Level 0:
        entity -> memberOf -> community

    Higher levels:
        child_community -> memberOf -> parent_community

    This produces an RDF representation of the hierarchy without
    creating a separate named graph for every community.
    """

    if dataset is None:
        dataset = Dataset()

    community_graph = dataset.graph(
        JS_SOURCE.community
    )

    dataset.bind(
        "js_source",
        JS_SOURCE
    )

    # -----------------------------------------------------------------
    # Serialize communities
    # -----------------------------------------------------------------

    for level, communities in hierarchy.items():

        for community_id, data in communities.items():

            community_uri = data["uri"]

            # ---------------------------------------------------------
            # Community identity
            # ---------------------------------------------------------

            community_graph.add(
                (
                    community_uri,
                    RDF.type,
                    JS_SOURCE.Community,
                )
            )

            community_graph.add(
                (
                    community_uri,
                    RDFS.label,
                    Literal(
                        str(community_uri)
                    ),
                )
            )

            # ---------------------------------------------------------
            # Membership
            # ---------------------------------------------------------

            for member in data["members"]:

                if not isinstance(
                    member,
                    URIRef,
                ):
                    continue

                community_graph.add(
                    (
                        member,
                        JS_SOURCE.memberOf,
                        community_uri,
                    )
                )

    return dataset

##### B. Execution

In [ ]:
# --------------------------------------------------
# 1. Build global entity graph from the completed KG
# --------------------------------------------------

hierarchy, entity_graph = build_graph_communities(
    dataset,
    max_levels=5,
    resolution=1.0
)


# --------------------------------------------------
# 2. Add community hierarchy to the Dataset
# --------------------------------------------------

dataset = serialize_communities(hierarchy, dataset)


# --------------------------------------------------
# 3. Serialize the completed KG
# --------------------------------------------------

dataset.serialize(
    "jurisynth_with_communities.nq",
    format="nquads"
)

<Graph identifier=Ne8c1d1e0bd4445888609c653f82b633a (<class 'rdflib.graph.Dataset'>)>